# Macaque MRI Atlas — end-to-end pipeline (v3)

Registration through to a smoothed 3-D surface atlas, in one notebook.
Cell headers cite `pipeline_tree_v3.txt` node numbers, so a node number is a search key.
Every tunable value lives in cell 1 (`0 SETUP AND PARAMETERS`); no later cell assigns one.


In [ ]:
# =====================================================================================
# 0  SETUP AND PARAMETERS
# =====================================================================================
# End-to-end pipeline, nodes 0 -> 6. Self-contained: builds its own template and
# registration under WORK, then parcellates, exports, fuses, mirrors and meshes.
# EVERY tunable value in this notebook is set in this cell. No later cell assigns one.
# 0.1  mount Drive and install dependencies
from google.colab import drive; drive.mount('/content/drive')
!pip -q install antspyx nibabel numpy scipy shapely matplotlib scikit-image svgpathtools plotly trimesh

import os, glob, json, time, pickle, subprocess, sys, csv, colorsys, re, shutil, hashlib
import numpy as np
import nibabel as nib
import ants
from pathlib import Path

# =====================================================================================
# 0.4.1  PATHS AND RUN IDENTITY
# =====================================================================================
DRIVE_ROOT  = "/content/drive/My Drive/macaque_atlas"
WORK        = f"{DRIVE_ROOT}/work_TESTv5"   # must match REGISTRATION_ONLY / TEST / Runner
RUN_NAME    = "demo1"                       # sub-folder under WORK for this run's outputs
MODULES_DIR = f"{DRIVE_ROOT}/modules"
TPL_DIR     = f"{DRIVE_ROOT}/template"
TEMPLATE    = "DB09"
PARCELLATION_LEVEL = "full"
SLICE_AXIS  = 1        # coronal
LR_AXIS     = 0        # after canonical reorient, axis 0 = R/L

# =====================================================================================
# 0.4.2  SUBJECTS                                                            [1.2.1, 1.2.2]
# =====================================================================================
# Single source of truth for subject ids and paths; must be byte-identical to the same
# block in REGISTRATION_ONLY, TEST and the Experiment Runner. A _zfix scan supersedes the
# un-fixed file for that subject; the two are never mixed.
NIFTI_OUT      = f"{DRIVE_ROOT}/scans/nifti_out"
NIFTI_OUT_CORR = f"{DRIVE_ROOT}/scans/nifti_out_corr"
LOCAL_SCANS    = "/content/scans"
DRIVE_SCANS = {
    "NHP1": f"{NIFTI_OUT}/NHP1/NHP1_scan9_reco1_zfix_skullstripped_cropped_final.nii.gz",
    "NHP2": f"{NIFTI_OUT}/NHP2/NHP2_scan3_reco1_skullstripped_cropped_final_trim.nii.gz",
}
# 1.2.4 provenance only; not consumed downstream. None = that subject has no mask file.
DRIVE_MASKS = {
    "NHP1": f"{NIFTI_OUT}/NHP1/NHP1_scan9_mask_final_zfix.nii",
    "NHP2": f"{NIFTI_OUT}/NHP2/NHP2BrainMask-final_frm.nii",
}

# =====================================================================================
# 0.4.3  REGISTRATION                                                          [2.1 - 2.3]
# =====================================================================================
REBUILD_TEMPLATE = False      # 1.1 / 2.1  force a template rebuild even if outputs exist
REG_TRANSFORM    = "Affine"   # 2.1.2.2  "Affine" (12 dof) | "TRSAA" | "SyN"
REG_METRIC       = "mattes"   # 2.1.2.3  MMI
FORCE_REREGISTER = False      # 2.1.2  ignore the cache
NCC_GATE         = 0.5        # 2.2.2  per-subject pass threshold
TARGET_ZOOMS          = (0.15, 0.75, 0.15)   # 2.3.1  (z0, z1, z2) mm on template axes
TPL_SLICES_TO_CONVERT = (60, 140, 249)       # 2.3.5  template indices to print
BG_THRESHOLD          = 0.0                  # 2.2.6.3  voxels > this count as brain

# =====================================================================================
# 0.4.4  PARCELLATION                                                            [3.1.x]
# =====================================================================================
HEMISPHERE        = "left"   # 3.1.1  "whole" | "left" | "right"
MIN_REGION_VOXELS = 6        # 3.1.3.1
SIMPLIFY_TOL      = 0.25     # 3.1.4.2  px, Douglas-Peucker tolerance
SMOOTH_BOUNDARIES = True     # 3.1.4.1
SMOOTH_SIGMA      = 2.0      # 3.1.4.1  px, Gaussian low-pass on the traced boundary
TRACE_MAX_HANDLES = None     # 3.1.4.3  None = keep the DP vertices; an int caps handles
SLICE_SUBSET      = None     # 3.1.7  None = every non-empty slice; (start, stop) to limit

# =====================================================================================
# 0.4.5  EXPORT AND QA                                                       [3.2.x, 4.4]
# =====================================================================================
LABEL_DETAIL       = "abbrev"   # 3.2.3.3  "none" | "abbrev" | "full"
LINE_MODE          = "single"   # 3.2.3.4  "single" | "double"
LINE_DOUBLE_OFFSET = 0.35       # 3.2.3.4
FLIP_LR              = False    # 3.2.1
LABEL_ALL_INSTANCES  = True     # 3.2.3.3
MIN_COMPONENT_VOXELS = 8        # 3.2.3.3
LABEL_FIT            = 0.60     # 3.2.3.3
LABEL_HEIGHT_FRAC    = 0.9      # 3.2.3.3
LABEL_FS_MAX         = 2.2      # 3.2.3.3
LABEL_FS_MIN         = 0.55     # 3.2.3.3
LABEL_OUTLINE_FRAC   = 0.16     # 3.2.3.3
EXPORT_SMOOTH_SIGMA  = 0.0      # 3.2.3.2  0 = export the traced geometry unchanged
EXPORT_SVGS    = True           # 3.2.5  False = skip export entirely; fusion reads graphs
EXPORT_EVERY_N = 10             # 3.2.5  1 = the full QA set
EXPORT_STROKE  = 0.36           # 3.2.3.2
AUTO_QA_COPY   = True           # 3.2.7  copy each export into RUN_DIR/qc

# =====================================================================================
# 0.4.6  FUSION                                                                    [5.x]
# =====================================================================================
# Units are VOXELS; 1 voxel == 1 SVG px. Tuning order: kp_alpha -> fit_tol_px ->
# node_match_max (from the measured S3_node_disp_p95_px) -> curv_lambda / kp_gamma.
FUSION_PARAMS_KW = dict(
    keypoints      = True,    # 5.6.4
    kp_alpha       = 4.0,     # px. Scale of preserved detail; tune first.
    kp_curv_thresh = 0.05,    # 1/px. Both signs kept (SATM CURV).
    kp_dmax        = 5.0,     # px. Beyond this, key points are rejected.
    kp_gamma       = 1/3,
    kp_l_scale     = None,
    fit_tol_px  = 0.05,       # 5.6.6.1  max chord deviation; drives the point count
    n_min_seg   = 2,
    n_max_seg   = 200,
    curv_lambda = 0.5,        # 5.6.6.2
    curv_sigma_px  = None,    # 5.6.3  None -> min(1.0, kp_alpha/4)
    curv_sample_px = 0.25,
    node_tol        = 1.0,    # 5.2.2.1
    node_match_max  = 35.0,   # 5.5.1.2
    strict_topo     = False,  # 5.5.3.1
    strict_topo_min = 0.98,
    orphan_policy   = "passthrough",   # 5.6.1
    min_arc_support = 1,
    reference_sid   = None,   # 5.6.1.1  None = the auto ladder (per slice)
    phantom_eps_px     = 1e-3,   # 5.3.3.1.2
    halt_on_situation3 = True,   # 5.3.3.4.3
    label_all_fragments = True,  # 5.6.4.4
    representation = "polyline",  # 5.6.2
    spline_smooth  = 0.0,
    loop_align     = "fft",       # 5.6.2.1
    outer_code         = 0,       # 5.7.3.1
    unlabeled_id_start = 8001,    # 5.7.3.4
    new_id_start       = 9001,    # 4.4.5
    node_arcs          = True,    # 5.7.2.2
    repair_dangling    = True,    # 5.7.1.3
    dangle_repair_px   = 1.5,
    min_face_area      = 0.0,
    build_polygons        = True,   # 5.7  the volumetric stage needs the polygons
    diag_ignore_unlabeled = True,   # 5.8.2.5
    flatten_px   = 0.05,      # 4.4.4  must be <= fit_tol_px
    grid_write   = None,
    svg_decimals = 4,
    run_comparative_diag = True,   # 5.8.2  forces one atlas rebuild PER SUBJECT
    metrics_avg_gl  = True,
    metrics_curv_ks = True,
    metrics_skele   = False,
)
FUSION_INPUT_PATHS = {}       # 5.1.2  optional per-subject overrides {sid: svg_path}
FUSION_WEIGHTS     = None     # 5.2.1  None = equal weights
FUSION_RUN_ID      = 1        # 5.7.4  bump each time fusion is re-run on new QA output
USE_SVG_INPUTS     = False    # 5.1.2  True = fuse the QA-edited SVGs instead of the graphs

# =====================================================================================
# 0.4.7  MIRROR, RASTERISE AND SURFACE                                             [6.x]
# =====================================================================================
MIRROR_WELD_TOL = 0.75    # 6.1.2.1  voxels; nodes this close to the midline snap onto it

ATLAS_PREVIEW_FILL   = 0.60          # 6.2.7
ATLAS_PREVIEW_SCALE  = 2.0           # 6.2.7  on-screen zoom only
UNDERLAY_SUBJECT     = "NHP1"        # 6.2.7.1  subject id, or None for the DB09 template
EXPORT_FILLED_SVGS   = False         # 6.2.7  True = also write SVGs
EXPORT_ATLAS_EVERY_N = 20            # 6.2.7
EXPORT_VARIANTS      = ("colored", "normal")   # 6.2.7.2

VOL_NAME        = "macaque_atlas"    # 6.3  -> <VOL_NAME>.spr / .sdt
M3C_SLICE_RANGE = None    # 6.4.3.1  None = all slices; (start, end) 1-based
M3C_CAP         = 0       # 6.4.3.2
M3C_LINEARIZE   = 0       # 6.4.3.3
M3C_CENTER      = 1       # 6.4.3.4
M3C_XY_SPACING  = 2       # 6.4.3.5
M3C_Z_SPACING   = 1       # 6.4.3.5
M3C_ZIP   = f"{DRIVE_ROOT}/M3C2012.zip"   # 6.4.1  zip on Drive, or an extracted folder
M3C_BUILD = "/content/m3c_build"          # 6.4.1  local build dir (not Drive)
M3C_RUN   = "/content/m3c_run"            # 6.4.2  local scratch: sdt/spr in, vtk out
COPY_VTK_TO_DRIVE = True                  # 6.4.5

PALETTE_CSV   = f"{DRIVE_ROOT}/reference_palette.csv"   # 6.6.2  optional rgb overrides
SMOOTH_ITERS  = 40        # 6.5.2  Taubin iterations. 20 = 2.9 deg mean dihedral, 40 = 1.4
SIZE_LIMIT_MB = 280       # 6.6.5.5
VIEWER_TITLE  = "Macaque atlas - hover a region for its name"   # 6.6.5.5
SHOW_INLINE   = False     # 6.6.5.5  a full-res mesh inline can hang Colab
PREVIEW_TRIS  = 250_000   # 6.6.6  inline preview budget
PREVIEW_SCALE = None      # 6.6.6  None = auto-pick a cluster cell to hit PREVIEW_TRIS

# =====================================================================================
# 0.4.8  DIAGNOSTICS
# =====================================================================================
VERIFY_SLICE = None       # 5.8.3  None = the representative slice

# =====================================================================================
# 0.2  MODULES
# =====================================================================================
if MODULES_DIR not in sys.path:
    sys.path.insert(0, MODULES_DIR)
for _m in ("boundary_graph.py", "edge_fusion.py"):
    if not os.path.exists(f"{MODULES_DIR}/{_m}"):
        raise FileNotFoundError(f"{_m} not found in {MODULES_DIR}.")
import boundary_graph as bg
import edge_fusion as ef

FUSION_PARAMS = ef.FusionParams(**FUSION_PARAMS_KW)   # 5.1.1

# =====================================================================================
# 0.3  DERIVED PATHS AND THE RUN TREE
# =====================================================================================
def _safe_dirname(s: str) -> str:
    """Reduce a run name to characters that are safe as a directory on Drive. [0.3.3]"""
    return re.sub(r"[^A-Za-z0-9._+-]", "_", s) if s else "default"


def _find_one(pattern, what="file"):
    """First path matching pattern; raises if nothing matches. [0.3]"""
    hits = sorted(glob.glob(pattern, recursive=True))
    if not hits:
        raise FileNotFoundError(f"nothing matched ({what}): {pattern}")
    return hits[0]


RUN_DIR = f"{WORK}/{_safe_dirname(RUN_NAME)}"
os.makedirs(TPL_DIR, exist_ok=True); os.makedirs(LOCAL_SCANS, exist_ok=True)

DB09_DIR        = f"{TPL_DIR}/template_db09"
DB09_MRI_RAW    = _find_one(f"{DB09_DIR}/**/mri_rb.nii.gz",   "DB09 raw MRI")
DB09_LABELS_RAW = _find_one(f"{DB09_DIR}/**/atlas_rb.nii.gz", "DB09 raw labels")
DB09_RGB2ACR    = _find_one(f"{DB09_DIR}/**/rgb2acr.json",    "DB09 rgb2acr")
DB09_ACR2FULL   = _find_one(f"{DB09_DIR}/**/acr2full.json",   "DB09 acr2full")

# 1.1 outputs (also globbed by the TEST notebook)
TEMPLATE_MRI_SQ = f"{DB09_DIR}/mri_rb_fullbrain_sq.nii.gz"
TEMPLATE_LABELS = f"{DB09_DIR}/DB09_labels_{PARCELLATION_LEVEL}_full_sq.nii.gz"
TEMPLATE_LUT    = TEMPLATE_LABELS.replace(".nii.gz", "_LUT.csv")
TEMPLATE_T1     = TEMPLATE_MRI_SQ

REG_DIR  = f"{WORK}/reg"                     # 2.1.3
SUBJ_DIR = f"{WORK}/reg_subjgrid"            # 2.3
# 0.3.1  subject-grid MRI, labels and LUT written by 2.3
SUBJGRID_MRI    = f"{SUBJ_DIR}/mri_in_subjgrid.nii.gz"
SUBJGRID_LABELS = f"{SUBJ_DIR}/labels_in_subjgrid.nii.gz"
SUBJGRID_LUT    = SUBJGRID_LABELS.replace(".nii.gz", "_LUT.csv")
SUBJGRID_JSON   = f"{SUBJ_DIR}/grid.json"

FUSION_INPUT_DIR = f"{RUN_DIR}/qc"                            # 5.1.2
REGISTRY_PATH    = f"{RUN_DIR}/atlas/region_registry.json"    # 5.7.4
VOL_DIR    = f"{RUN_DIR}/volume"                              # 6.3
VTK_OUT    = f"{RUN_DIR}/vtk"                                 # 6.4.5
FINAL_DIR  = f"{RUN_DIR}/finalized_product"                   # 6.6
NIFTI_DIR  = f"{FINAL_DIR}/nifti"
MESH_DIR   = f"{FINAL_DIR}/meshes"
VIEWER_DIR = f"{FINAL_DIR}/viewer"
SLICER_DIR = f"{FINAL_DIR}/slicer"                            # 6.6.6

# 0.3.3  per-run scratch tree under RUN_DIR (state, lines, atlas, qc)
for d in [REG_DIR, SUBJ_DIR, f"{WORK}/state", f"{WORK}/qc",
          f"{RUN_DIR}/state", f"{RUN_DIR}/lines_propagated",
          f"{RUN_DIR}/lines_export_for_qa/slices", f"{RUN_DIR}/lines_export_for_qa/tables",
          f"{RUN_DIR}/atlas", f"{RUN_DIR}/qc", VOL_DIR, VTK_OUT,
          NIFTI_DIR, MESH_DIR, VIEWER_DIR, SLICER_DIR]:
    os.makedirs(d, exist_ok=True)


def save_state(name, obj):
    """Persist one pipeline object under RUN_DIR/state/. [0.3.3]"""
    with open(f"{RUN_DIR}/state/{name}.pkl", "wb") as f:
        pickle.dump(obj, f)


def save_work_state(name, obj):
    """WORK-level state ('preproc', 'regs', 'regs_subjgrid') -> WORK/state/. Built once
    per WORK and read by every run inside it. [0.3.3]"""
    with open(f"{WORK}/state/{name}.pkl", "wb") as f:
        pickle.dump(obj, f)


def load_state(name, default=KeyError):
    """Lookup ladder: this run's state, then the shared work/ cache (which holds the
    per-notebook 'regs' and 'preproc'). First hit wins. [0.3.3]"""
    cands = [f"{RUN_DIR}/state/{name}.pkl", f"{WORK}/state/{name}.pkl"]
    path = next((p for p in cands if os.path.exists(p)), None)
    if path is None:
        if default is KeyError:
            raise FileNotFoundError(f"state '{name}' not found (tried: {cands})")
        return default
    with open(path, "rb") as f:
        return pickle.load(f)


def report_outputs(cell_name, files=(), state_keys=(), checks=()):
    """Print the files, state keys and gate results a cell produced. [0.3.3]"""
    print(f"\n=== {cell_name}: outputs ===")
    for f in files:
        ok = os.path.exists(f); sz = os.path.getsize(f) if ok else 0
        print(f"  {'OK ' if ok and sz>0 else '!! '}{f}  ({sz} bytes)")
    for k in state_keys:
        p = f"{RUN_DIR}/state/{k}.pkl"
        print(f"  {'OK ' if os.path.exists(p) else '!! '}state/{k}.pkl")
    for label, okc, hint in checks:
        print(f"  {'PASS' if okc else 'WARN'}: {label}" + ("" if okc else f"  -> {hint}"))
    print("=" * (len(cell_name) + 16))


print("Setup done. Template =", TEMPLATE)
print("  WORK     :", WORK)
print("  RUN_DIR  :", RUN_DIR)
print("  subjects :", list(DRIVE_SCANS))
print(f"  registration: {REG_TRANSFORM}/{REG_METRIC}  NCC gate {NCC_GATE}  grid {TARGET_ZOOMS}")
print(f"  parcellation: hemi={HEMISPHERE} simplify_tol={SIMPLIFY_TOL} "
      f"smooth_sigma={SMOOTH_SIGMA if SMOOTH_BOUNDARIES else 0} subset={SLICE_SUBSET}")
print(f"  fusion: kp_alpha={FUSION_PARAMS.kp_alpha} fit_tol_px={FUSION_PARAMS.fit_tol_px} "
      f"build_polygons={FUSION_PARAMS.build_polygons}")


## Stage 1 - Preprocessing

Scan preparation happens outside this notebook: Bruker to NIfTI conversion, contrast
correction in ITK-SNAP, skull-stripping and initial segmentation in 3D Slicer, orientation
correction, manual refinement in ITK-SNAP, and cropping the original scan with the mask.

The two cells below build the DB09 template that everything registers to, and stage the
already-preprocessed scans.

In [ ]:
# =====================================================================================
# 1.1  BUILD THE DB09 TEMPLATE
#
# DB09 ships one hemisphere. It is mirrored across its medial (x~0) face into a full
# symmetric brain and padded to a square in-plane grid (M3C requires nRow == nCol).
# Cached: skips the rebuild when all outputs are already present.
# =====================================================================================
import os, json, csv, colorsys
import numpy as np, nibabel as nib

# 1.1  mirror the DB09 hemisphere into a full symmetric brain
def _build_full_brain(vol, affine, lr_axis=LR_AXIS):
    """Mirror the DB09 hemisphere across its medial (x~0) face."""
    x0 = affine[0, 3]; dx = affine[0, 0]; n = vol.shape[lr_axis]
    x_start, x_end = x0, x0 + dx * (n - 1)
    at_high_idx = abs(x_end) < abs(x_start)          # medial face at the high index?
    flipped = np.flip(vol, axis=lr_axis)
    if at_high_idx:
        full = np.concatenate([vol, flipped], axis=lr_axis); new_x0 = x0
    else:
        full = np.concatenate([flipped, vol], axis=lr_axis); new_x0 = x0 - dx * n
    new_aff = affine.copy(); new_aff[0, 3] = new_x0
    return full, new_aff

# 6.3.1  in-plane grid made square here; re-asserted before the SPR write
def _pad_square_inplane(vol, slice_axis, affine):
    """Pad the two in-plane axes to a common size."""
    inplane = [a for a in range(3) if a != slice_axis]
    target  = max(vol.shape[a] for a in inplane)
    pad = [[0, 0], [0, 0], [0, 0]]; shift = np.zeros(3)
    for a in inplane:
        diff = target - vol.shape[a]; before = diff // 2
        pad[a] = [before, diff - before]; shift[a] = before
    out = np.pad(vol, pad, mode="constant", constant_values=0)
    new_aff = affine.copy(); new_aff[:3, 3] = affine[:3, 3] - affine[:3, :3] @ shift
    return out, new_aff

def _build_lut_from_jsons(rgb2acr, acr2full):
    """rgb2acr.json -> acr2full.json chained into {id: {abbrev, name, rgb_hex}}."""
    lut = {}
    for idx, (hexcol, acr) in enumerate(rgb2acr.items()):
        if acr == "[-]" or hexcol.upper() == "000000":
            continue
        full = acr2full.get(acr, "")
        if (not acr) or acr.startswith("["):
            abbrev = acr if acr else f"id{idx}"
            name   = full if full else (acr if acr else f"unnamed_{idx}")
        else:
            abbrev = acr; name = full if full else acr
        lut[idx] = {"abbrev": abbrev, "name": name, "rgb_hex": "#" + hexcol.upper()}
    return lut

def _det_hex(rid):
    """Golden-ratio hue fallback for an id with no DB09 colour."""
    h = (int(rid) * 0.61803398875) % 1.0
    r, g, b = colorsys.hsv_to_rgb(h, 0.65, 0.92)
    return "#{:02x}{:02x}{:02x}".format(round(r*255), round(g*255), round(b*255))

# 1.1  build the template MRI, labels and LUT
def build_db09_template():
    have_all = all(os.path.exists(p) for p in
                   (TEMPLATE_MRI_SQ, TEMPLATE_LABELS, TEMPLATE_LUT))
    if have_all and not REBUILD_TEMPLATE:
        print("template: cached full-brain squared files present, skipping rebuild.")
        return
    print("template: mirror -> square -> save")
    lab_img = nib.as_closest_canonical(nib.load(DB09_LABELS_RAW))
    mri_img = nib.as_closest_canonical(nib.load(DB09_MRI_RAW))
    merged  = np.asanyarray(lab_img.dataobj).astype(np.int32); ref_affine = lab_img.affine.copy()
    mri_np  = np.asanyarray(mri_img.dataobj).astype(np.float32); mri_aff = mri_img.affine.copy()

    merged, ref_affine = _build_full_brain(merged, ref_affine)
    mri_np, mri_aff    = _build_full_brain(mri_np, mri_aff)
    merged, ref_affine = _pad_square_inplane(merged, SLICE_AXIS, ref_affine)
    mri_np, mri_aff    = _pad_square_inplane(mri_np, SLICE_AXIS, mri_aff)

    nib.save(nib.Nifti1Image(mri_np, mri_aff), TEMPLATE_MRI_SQ)
    nib.save(nib.Nifti1Image(merged.astype(np.int32), ref_affine), TEMPLATE_LABELS)

    with open(DB09_RGB2ACR)  as f: rgb2acr  = json.load(f)
    with open(DB09_ACR2FULL) as f: acr2full = json.load(f)
    lut = _build_lut_from_jsons(rgb2acr, acr2full)
    with open(TEMPLATE_LUT, "w", newline="") as f:
        w = csv.writer(f); w.writerow(["id", "abbreviation", "name", "color_hex", "type"])
        for k in sorted(lut):
            e = lut[k]; nm = e["name"]
            kind = ("white_matter" if "white matter" in nm.lower()
                    else "ventricle" if "ventricle" in nm.lower() else "gray/other")
            w.writerow([k, e["abbrev"], nm, e.get("rgb_hex") or _det_hex(k), kind])
    print(f"  built {merged.shape}, {len(np.unique(merged))-1} regions")

# 1.1  verify the template outputs
def verify_template():
    """Gate: labels and MRI on one grid, square in-plane, labels present, L-R symmetric."""
    lab = nib.load(TEMPLATE_LABELS); t1 = nib.load(TEMPLATE_MRI_SQ)
    D0, D1, D2 = lab.shape
    arr = np.asanyarray(lab.dataobj)
    ids = np.unique(arr); ids = ids[ids > 0]
    labm = arr > 0
    sym = float((arr[labm] == np.flip(arr, LR_AXIS)[labm]).mean()) if labm.any() else float("nan")
    zooms = tuple(round(float(z), 4) for z in lab.header.get_zooms()[:3])
    print(f"  labels : {lab.shape} @ {zooms} mm, {ids.size} regions")
    print(f"  MRI    : {t1.shape}")
    print(f"  labeled-voxel L-R mirror match: {sym*100:.2f}%  (want ~100%)")
    checks = [
        ("labels and MRI share one grid", lab.shape == t1.shape, "rebuild the template"),
        (f"in-plane grid is SQUARE ({D0}x{D2})", D0 == D2, "M3C needs nRow == nCol"),
        ("labels present", ids.size > 1, "empty label volume"),
        ("full brain is L-R symmetric", sym > 0.98,
         "the wrong medial face was mirrored -- set REBUILD_TEMPLATE=True"),
    ]
    report_outputs("CELL 1 template",
                   files=[TEMPLATE_MRI_SQ, TEMPLATE_LABELS, TEMPLATE_LUT], checks=checks)
    if not all(ok for _, ok, _ in checks):
        raise RuntimeError("template invalid -- set REBUILD_TEMPLATE=True and re-run")

# 0.3.2  rebuild combined_lut (id -> abbreviation / name / rgb / kind) from the LUT CSV
def load_combined_lut(lut_csv=None):
    """{id: {abbrev, name, rgb_hex, kind}} rebuilt from the LUT CSV."""
    lut_csv = lut_csv or TEMPLATE_LUT
    out = {}
    with open(lut_csv, newline="") as f:
        for row in csv.DictReader(f):
            try: rid = int(float(row["id"]))
            except (KeyError, TypeError, ValueError): continue
            out[rid] = {"abbrev": row.get("abbreviation", str(rid)),
                        "name":   row.get("name", str(rid)),
                        "rgb_hex": row.get("color_hex", ""),
                        "kind":   row.get("type", "region")}
    return out

build_db09_template()
verify_template()
combined_lut = load_combined_lut()
print(f"\n  combined_lut: {len(combined_lut)} regions")


In [ ]:
# =====================================================================================
# 1.2  STAGE THE PREPROCESSED SCANS
#
# 1.1.1-1.1.4 happen outside this notebook (ITK-SNAP / 3D Slicer). This cell stages
# the results.
# 1.2.3  each scan is copied to local disk once; everything downstream reads the copy
# 1.2.4  orientation report (axis labelling and voxel size per subject vs template)
# =====================================================================================
import os, time, shutil
import nibabel as nib

# SINGLE SOURCE OF TRUTH for subject ids and paths. Every later stage loops over whatever
# is here; no other cell changes to add or remove a subject.
# recorded for provenance and existence-checked so a missing or un-fixed mask is visible.
# Leave a value as None if that subject has no separate mask file.

os.makedirs(LOCAL_SCANS, exist_ok=True)

# ---- orientation flag ---------------------------------------------
# ANTs registers in physical space, so array axis order does not affect any later stage.
# A disagreement between subjects is worth seeing but is not a reason to stop.
# 1.2.4  orientation report: axis labelling and voxel size per subject vs template
def orientation_report(pp):
    """Print each subject's axis labelling and voxel size, and the template's for
    reference. Returns True if every subject shares one labelling."""
    codes = {}
    print("\naxis labelling (codes = direction each array axis points toward):")
    for sid, d in pp.items():
        img = nib.load(d["brain"])
        z   = tuple(round(float(v), 4) for v in img.header.get_zooms()[:3])
        codes[sid] = nib.aff2axcodes(img.affine)
        print(f"  {sid}: {codes[sid]}  zooms {z}")
    tpl = nib.load(TEMPLATE_MRI_SQ)
    tz  = tuple(round(float(v), 4) for v in tpl.header.get_zooms()[:3])
    print(f"  template: {nib.aff2axcodes(tpl.affine)}  zooms {tz}  "
          f"(canonical RAS from CELL 1; subjects are not expected to match it)")
    return len(set(codes.values())) <= 1

# 1.2.3  copy each scan to local disk once; every later stage reads the local copy
def stage_scans(drive_map=DRIVE_SCANS, local_dir=LOCAL_SCANS, verbose=True):
    """Copy each scan Drive -> local once. Idempotent: a staged file of matching size is
    left alone."""
    staged = {}
    for sid, src in drive_map.items():
        if not os.path.exists(src):
            raise FileNotFoundError(f"{sid}: scan not found on Drive: {src}")
        src_sz = os.path.getsize(src)
        if src_sz == 0:
            raise ValueError(f"{sid}: scan is empty on Drive: {src}")
        dst = f"{local_dir}/{os.path.basename(src)}"
        t0 = time.time()
        if os.path.exists(dst) and os.path.getsize(dst) == src_sz:
            if verbose: print(f"  {sid}: already staged ({src_sz/1e6:.1f} MB)")
        else:
            shutil.copyfile(src, dst)
            if verbose: print(f"  {sid}: staged {src_sz/1e6:.1f} MB  {time.time()-t0:.1f}s")
        staged[sid] = {"local": dst, "drive": src, "bytes": src_sz}
    return staged

# 1.2.1 / 1.2.2  single source of truth for subject ids and paths; a _zfix scan supersedes the un-fixed file
def import_preprocessed():
    """Header-check each staged scan and record it in the WORK-level 'preproc' state."""
    t_all = time.time()
    print("staging scans to local disk:")
    staged = stage_scans()
    pp, missing_masks = {}, []
    for sid, s in staged.items():
        img = nib.load(s["local"])                     # header-only sanity load
        mask = (DRIVE_MASKS or {}).get(sid)
        if mask and not os.path.exists(mask):
            missing_masks.append((sid, mask)); mask = None
        pp[sid] = {"brain": s["local"], "drive_src": s["drive"], "mask": mask,
                   "pre": s["local"], "warpedtpl": None,
                   "shape": tuple(int(v) for v in img.shape[:3]),
                   "zooms": tuple(round(float(z), 4) for z in img.header.get_zooms()[:3])}
    same_frame = orientation_report(pp)
    save_work_state("preproc", pp)
    checks = [("every subject has a mask on Drive", not missing_masks,
               f"create: {[m for _s, m in missing_masks]}"),
              ("all subjects share one axis labelling", same_frame,
               "not fatal (registration is done in physical space), but a mismatch "
               "usually means one scan was exported in a different frame")]
    report_outputs("CELL 2 import", files=[d["brain"] for d in pp.values()], checks=checks)
    print(f"\nImported {len(pp)} subject(s) in {time.time()-t_all:.1f}s:")
    for sid, d in pp.items():
        print(f"  {sid}: {d['shape']} @ {d['zooms']} mm")
        print(f"        {os.path.basename(d['drive_src'])}")
    return pp

SUBJECT_SCANS = {sid: d["local"] for sid, d in stage_scans(verbose=False).items()}
import_preprocessed()


## Stage 2 - Initial registration

Affine subject-to-template with ANTs, driven by Mattes mutual information, scored and gated
per subject; then a collapse onto one shared working grid at the acquired through-plane
resolution.

In [ ]:
# =====================================================================================
# 2.1 / 2.2  AFFINE REGISTRATION: subject -> template
#
# 2.1.1  confirm all files exist
# 2.1.2  estimate transform per subject (coarse-to-fine rigid+affine, MMI)
# 2.1.3  emit registration products on the template grid
# 2.1.4  a template fingerprint is stored beside each transform; a cache built against
#        a different template is rejected instead of silently reused
# 2.2    score and gate (NCC, NMI)
# =====================================================================================
import os, pickle, shutil, hashlib, time
import numpy as np, nibabel as nib, ants
from tqdm.auto import tqdm

# 2.1.4.1  fingerprint of the template being registered TO
def _template_fp():
    """Identity of the template being registered to."""
    im = nib.load(TEMPLATE_MRI_SQ)
    return {"file":   os.path.basename(TEMPLATE_MRI_SQ),
            "shape":  tuple(int(s) for s in im.shape[:3]),
            "zooms":  tuple(round(float(z), 5) for z in im.header.get_zooms()[:3]),
            "affine": hashlib.md5(np.round(im.affine, 6).tobytes()).hexdigest(),
            "xform":  REG_TRANSFORM}

def _persist(src, stem):
    """Copy one ANTs transform out of /tmp into work/reg/, keeping its extension."""
    ext = ".nii.gz" if src.endswith(".nii.gz") else os.path.splitext(src)[1]
    dst = f"{REG_DIR}/{stem}{ext}"
    if os.path.abspath(src) != os.path.abspath(dst):
        shutil.copy(src, dst)
    return dst

# 2.2.1 / 2.2.3  NCC and foreground-masked NMI, warped subject vs template
def registration_metrics(warped_path, template_path=None):
    """Score: normalized cross-correlation, plus normalised mutual information from a
    foreground-masked joint histogram."""
    template_path = template_path or TEMPLATE_T1
    f = ants.image_read(template_path); m = ants.image_read(warped_path)
    fa, ma = f.numpy().ravel(), m.numpy().ravel()
    a = (fa - fa.mean()) / (fa.std() + 1e-9); b = (ma - ma.mean()) / (ma.std() + 1e-9)
    ncc = float(np.mean(a * b))
    fg = (fa > 0) | (ma > 0)
    hist, _, _ = np.histogram2d(fa[fg], ma[fg], bins=64)
    pxy = hist / (hist.sum() + 1e-9); px = pxy.sum(1); py = pxy.sum(0)
    Hx  = -np.sum(px[px > 0] * np.log(px[px > 0]))
    Hy  = -np.sum(py[py > 0] * np.log(py[py > 0]))
    Hxy = -np.sum(pxy[pxy > 0] * np.log(pxy[pxy > 0]))
    return {"ncc": ncc, "nmi": float((Hx + Hy) / (Hxy + 1e-9))}

# 2.1  affine transform subject -> template (ANTs)
def register_subjects(force=FORCE_REREGISTER, transform=REG_TRANSFORM):
    """Affine-register every subject to the template; forward and inverse transforms and
    the warped image are written on the TEMPLATE grid."""
    pp = load_state("preproc")
    fp = _template_fp()
    print(f"registering to {TEMPLATE_MRI_SQ}")
    print(f"  shape {fp['shape']}  zooms {fp['zooms']}  transform {transform}\n")
    # 2.1.1  confirm all files exist
    for sid, d in pp.items():                       # confirm all files exist first
        if not (os.path.exists(d["brain"]) and os.path.getsize(d["brain"]) > 0):
            raise FileNotFoundError(f"{sid}: scan missing or empty: {d['brain']}")
        nib.load(d["brain"])
    fixed = ants.image_read(TEMPLATE_MRI_SQ)
    regs  = {}
    bar = tqdm(pp.items(), desc="register", unit="subj")
    for sid, d in bar:
        warped = f"{REG_DIR}/{sid}_in_template.nii.gz"
        tfm    = f"{REG_DIR}/{sid}_transforms.pkl"
        if not force and os.path.exists(warped) and os.path.exists(tfm):
            try:
                saved = pickle.load(open(tfm, "rb"))
                same  = saved.get("template_fp") == fp
                files = all(os.path.exists(p) for p in saved.get("fwd", []) + saved.get("inv", []))
                if same and files:
                    regs[sid] = {**saved, "warped": warped, "seconds": 0.0, "cached": True}
                    bar.set_postfix_str(f"{sid}: cached"); continue
                # 2.1.4.2  a cache built against a different template is rejected, not reused
                reason = "different template" if not same else "transform files missing"
                print(f"  {sid}: cache REJECTED -- {reason}")
            except Exception as e:
                print(f"  {sid}: cache unreadable ({e}) -> re-registering")
        bar.set_postfix_str(f"{sid}: registering")
        t0 = time.time()
        # 2.1.2  estimate the transform per subject (2.1.2.1 initial alignment, 2.1.2.2 coarse-to-fine rigid+affine, 2.1.2.3 MMI)
        reg = ants.registration(fixed=fixed, moving=ants.image_read(d["brain"]),
                                type_of_transform=transform, aff_metric=REG_METRIC)
        dt = time.time() - t0
        # 2.1.3  emit registration products on the template grid
        ants.image_write(reg["warpedmovout"], warped)
        # 2.1.3.1  forward and inverse transforms recorded
        entry = {"fwd": [_persist(p, f"{sid}_fwd{k}") for k, p in enumerate(reg["fwdtransforms"])],
                 "inv": [_persist(p, f"{sid}_inv{k}") for k, p in enumerate(reg["invtransforms"])],
                 "template_fp": fp}
        with open(tfm, "wb") as f:
            pickle.dump(entry, f)
        regs[sid] = {**entry, "warped": warped, "seconds": dt, "cached": False}
        bar.set_postfix_str(f"{sid}: {dt:.0f}s")

    # 2.2.5  save transforms, warped scans, elapsed time and cache status
    save_work_state("regs", regs)
    lab_shape = nib.load(TEMPLATE_LABELS).shape
    checks = []
    for sid, r in regs.items():
        # 2.2  score and gate
        m = registration_metrics(r["warped"], TEMPLATE_MRI_SQ)
        checks.append((f"{sid} template overlap (NCC={m['ncc']:.2f}, NMI={m['nmi']:.2f})",
                       # 2.2.2  pass or warn per subject against NCC_GATE
                       m["ncc"] > NCC_GATE,
                       "low overlap -> check skull-stripping and orientation, or try TRSAA/SyN"))
        checks.append((f"{sid} warped shape == label shape {lab_shape}",
                       # 2.2.4  confirm warped shape == label shape
                       nib.load(r["warped"]).shape == lab_shape,
                       "registration wrote a mismatched grid"))
    report_outputs("CELL 3 register", files=[r["warped"] for r in regs.values()], checks=checks)
    n_new = sum(0 if r["cached"] else 1 for r in regs.values())
    print(f"\n{len(regs)} subject(s): {n_new} registered, {len(regs)-n_new} from cache.")
    for sid, r in regs.items():
        status = "cached" if r["cached"] else f"{r['seconds']:.0f}s"
        print(f"  {sid}: {r['warped']}  ({status})")
    return regs

register_subjects()


In [ ]:
# =====================================================================================
# 2.3  COLLAPSE TO THE SHARED WORKING GRID
#
# 2.3.1  target voxel size; 2.3.2 build the reference grid
# 2.3.3  resample the template labels with an identity transform
# 2.3.4  resample each subject from the ORIGINAL scan through the 2.1 transform
# 2.3.5  record and verify the new grid
#
# After this cell TEMPLATE_LABELS / TEMPLATE_LUT / TEMPLATE_T1 point at the working
# grid, so every later stage reads the subject-grid products.
# =====================================================================================
import os, json, shutil
import numpy as np, nibabel as nib, ants

# scan headers: a badly oriented scan would then mis-assign the axes silently).
# 0.75 sits on SLICE_AXIS (=1, coronal) = the acquired slice thickness; the other two are
# the template's own in-plane spacing, so 1 voxel == 1 SVG px keeps its physical meaning.

def template_slice_to_subjgrid(idx, scale, n_slices=None):
    """Template slice index -> the slice at the same anatomical position on the new grid."""
    j = int(round(idx * scale))
    return max(0, min(j, n_slices - 1)) if n_slices else max(0, j)

# 2.3  collapse to the shared working grid
def resample_to_working_grid():
    """Resample template MRI (reference grid), template labels (nearest neighbour) and
    every subject (forward transform, single interpolation) onto the working grid."""
    regs = load_state("regs")
    # 2.3.1 / 2.3.1.1  target voxel size: in-plane kept, through-plane restored to the acquired thickness
    print(f"working grid on template axes: {TARGET_ZOOMS} mm "
          f"(through-plane on SLICE_AXIS={SLICE_AXIS})")

    tpl = ants.image_read(TEMPLATE_MRI_SQ)
    # 2.3.2 / 2.3.2.1  build the reference grid (2.3.2.2 new count = old count x old spacing / new spacing)
    ref = ants.resample_image(tpl, TARGET_ZOOMS, use_voxels=False, interp_type=0)   # 0 = linear
    ants.image_write(ref, SUBJGRID_MRI)

    # 2.3.3 / 2.3.3.1  template labels onto the reference grid with an identity transform
    lab_rs = ants.apply_transforms(fixed=ref, moving=ants.image_read(TEMPLATE_LABELS),
                                   transformlist=[], interpolator="nearestNeighbor")
    ants.image_write(lab_rs, SUBJGRID_LABELS)
    shutil.copyfile(TEMPLATE_LUT, SUBJGRID_LUT)

    pp = load_state("preproc")
    subj = {}
    for sid, r in regs.items():
        dst = f"{SUBJ_DIR}/{sid}_in_subjgrid.nii.gz"
        # 2.3.4 / 2.3.4.1  each subject from the ORIGINAL scan through the 2.1 forward transform: one interpolation
        w = ants.apply_transforms(fixed=ref, moving=ants.image_read(pp[sid]["brain"]),
                                  transformlist=r["fwd"], interpolator="linear")
        ants.image_write(w, dst)
        subj[sid] = dst

    tz    = tuple(round(float(z), 5) for z in nib.load(TEMPLATE_MRI_SQ).header.get_zooms()[:3])
    scale = tz[SLICE_AXIS] / TARGET_ZOOMS[SLICE_AXIS]
    meta  = {"target_zooms": list(TARGET_ZOOMS),
             "target_shape": [int(s) for s in ref.shape],
             "template_zooms": list(tz), "template_shape": [int(s) for s in tpl.shape],
             "slice_axis": SLICE_AXIS, "slice_index_scale": scale,
             "mri": SUBJGRID_MRI, "labels": SUBJGRID_LABELS, "lut": SUBJGRID_LUT,
             "subjects": subj}
    # 2.3.5  record the new grid
    with open(SUBJGRID_JSON, "w") as f:
        json.dump(meta, f, indent=2)

    regs_sg = {sid: {**r, "warped": subj[sid], "warped_template_grid": r["warped"]}
               for sid, r in regs.items()}
    save_work_state("regs_subjgrid", regs_sg)

    # ---- checks ----------------------------------------------------------------------
    a = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    b = nib.load(SUBJGRID_LABELS).get_fdata().astype(np.int32)
    ids_a = set(np.unique(a)) - {0}; ids_b = set(np.unique(b)) - {0}
    lost  = sorted(int(v) for v in ids_a - ids_b)
    D0, D1, D2 = b.shape
    thick = {sid: round(float(max(nib.load(d["brain"]).header.get_zooms()[:3])), 4)
             for sid, d in pp.items()}
    off   = {sid: v for sid, v in thick.items()
             if abs(v - TARGET_ZOOMS[SLICE_AXIS]) > 1e-3}
    # 2.3.5  verify the new grid and the registration outputs on it
    checks = [
        (f"labels and every subject share one grid {b.shape}",
         all(nib.load(p).shape == b.shape for p in subj.values()),
         "resample wrote mismatched grids"),
        (f"regions kept {len(ids_b)}/{len(ids_a)}", len(lost) == 0,
         f"{len(lost)} region(s) thinner than one {TARGET_ZOOMS[SLICE_AXIS]} mm slice: {lost[:12]}"),
        (f"in-plane grid still square ({D0}x{D2})", D0 == D2, "M3C needs nRow == nCol"),
        (f"acquired thickness == grid through-plane ({TARGET_ZOOMS[SLICE_AXIS]} mm)",
         not off, f"acquired {off} mm -- TARGET_ZOOMS no longer matches the scans"),
    ]
    for sid, p in subj.items():
        ncc = registration_metrics(p, SUBJGRID_MRI)["ncc"]
        checks.append((f"{sid} overlap on the working grid (NCC={ncc:.2f})", ncc > NCC_GATE,
                       "re-check CELL 3 for this subject"))
    report_outputs("CELL 4 resample -> working grid",
                   files=[SUBJGRID_MRI, SUBJGRID_LABELS, SUBJGRID_LUT] + list(subj.values()),
                   checks=checks)

    n_new = b.shape[SLICE_AXIS]
    print(f"\n  template grid : {tuple(int(s) for s in tpl.shape)} @ {tz} mm")
    print(f"  working grid  : {tuple(int(s) for s in ref.shape)} @ {meta['target_zooms']} mm")
    print(f"  slice index conversion: new = round(old x {scale:.4f}), valid 0..{n_new-1}")
    for old in TPL_SLICES_TO_CONVERT:
        print(f"    template slice {old:4d} -> working-grid slice "
              f"{template_slice_to_subjgrid(old, scale, n_new)}")
    print(f"  in-plane {TARGET_ZOOMS[0]:.4f} mm == template in-plane, so the pixel-denominated "
          f"fusion params (kp_alpha, fit_tol_px, SMOOTH_SIGMA, node_match_max, simplify_tol) "
          f"keep their physical size.")
    return regs_sg

regs_sg = resample_to_working_grid()

# Everything from CELL 5 on works on the WORKING GRID.
SUBJ_GRID       = json.load(open(SUBJGRID_JSON))
combined_lut    = load_combined_lut(TEMPLATE_LUT)
save_work_state("regs", regs_sg)     # downstream 'regs' now points at the working grid
print(f"\n  TEMPLATE_* rebound to the working grid; regs['warped'] -> *_in_subjgrid.nii.gz")


In [ ]:
# =====================================================================================
# 2.2.6  TRANSFORM REPORT   (diagnostic only: writes nothing)
#
# 2.2.6.1  affine decomposed on the TEMPLATE ARRAY AXES
# 2.2.6.2  per-axis voxel step; 2.2.6.3 occupancy extents
# 2.2.6.4  empirical volume scale vs the affine determinant; 2.2.6.5 obliquity
# =====================================================================================
import numpy as np, nibabel as nib, ants
from scipy.spatial.transform import Rotation as _Rot

_E_CACHE = {}

# 2.2.6.1  RQ decomposition helper: per-axis scale and shear
def _rq3(M):
    """RQ decomposition M = K @ Qo; diag(K) = per-axis scale, off-diagonals = shear."""
    P = np.flipud(M).T
    Qq, Rr = np.linalg.qr(P)
    K  = np.fliplr(np.flipud(Rr.T))
    Qo = np.flipud(Qq.T)
    for i in range(3):
        if K[i, i] < 0:
            K[:, i] *= -1.0; Qo[i, :] *= -1.0
    if np.linalg.det(Qo) < 0:
        K[:, -1] *= -1.0; Qo[-1, :] *= -1.0
    return K, Qo

# 2.2.6.3  occupancy extent of one volume (voxels above BG_THRESHOLD)
def _extent(path, thr=BG_THRESHOLD):
    """Per-axis bounding box of the non-background data, plus total brain volume."""
    if path in _E_CACHE:
        return _E_CACHE[path]
    img = nib.load(path)
    zm  = [float(z) for z in img.header.get_zooms()[:3]]
    shp = tuple(int(s) for s in img.shape[:3])
    m   = np.asanyarray(img.dataobj) > thr
    rows, nvox = [], int(m.sum())
    for a in (0, 1, 2):
        oa = tuple(b for b in (0, 1, 2) if b != a)
        nz = np.flatnonzero(m.any(axis=oa))
        if nz.size == 0:
            rows.append({"axis": a, "lo": -1, "hi": -1, "n": 0, "bg": shp[a], "mm": 0.0})
        else:
            lo, hi = int(nz.min()), int(nz.max()); n = hi - lo + 1
            rows.append({"axis": a, "lo": lo, "hi": hi, "n": n,
                         "bg": shp[a] - n, "mm": n * zm[a]})
    del m
    _E_CACHE[path] = {"rows": rows, "zooms": zm, "shape": shp,
                      "vol_mm3": nvox * float(np.prod(zm))}
    return _E_CACHE[path]

def _print_extent(tag, e):
    print(f"    {tag}: shape {e['shape']} spacing {tuple(round(z,4) for z in e['zooms'])}"
          f"  brain volume {e['vol_mm3']/1000.0:.2f} cm3")
    for r in e["rows"]:
        print(f"      axis {r['axis']}: data {r['lo']}..{r['hi']} = {r['n']} slices "
              f"({r['mm']:.2f} mm), background {r['bg']} of {e['shape'][r['axis']]}")

# 2.2.6  transform report
def transform_report(sid, regs, scans):
    """Decompose and report one subject's estimated affine."""
    print(f"\n================ {sid} ================")
    subj = ants.image_read(scans[sid])
    tpl  = ants.image_read(TEMPLATE_MRI_SQ)
    tx   = ants.read_transform(regs[sid]["fwd"][0])
    par  = np.asarray(tx.parameters, dtype=float)
    fxp  = np.asarray(tx.fixed_parameters, dtype=float)
    if par.size < 12:
        raise ValueError(f"{sid}: fwd[0] has {par.size} params, not a 3D affine")
    A = par[:9].reshape(3, 3)     # ITK: q_moving = A(p_fixed - c) + c + t
    t = par[9:12]; c = fxp[:3]
    B = np.linalg.inv(A)          # subject -> template

    D_s  = np.asarray(subj.direction, dtype=float).reshape(3, 3)
    D_t  = np.asarray(tpl.direction,  dtype=float).reshape(3, 3)
    sp_s = np.asarray(subj.spacing, dtype=float)
    sp_t = np.asarray(tpl.spacing,  dtype=float)
    Dt_i = np.linalg.inv(D_t)
    # 2.2.6.1  affine expressed on the template array axes
    Barr = Dt_i @ B @ D_t         # expressed on the template's array axes

    print("  --- AFFINE, subject -> template, on the TEMPLATE ARRAY AXES ---")
    detB = float(np.linalg.det(Barr))
    if detB < 0:
        print("  !! NEGATIVE DETERMINANT: this affine contains a REFLECTION. Check subject "
              "orientation before trusting anything downstream.")
    U, S, Vt = np.linalg.svd(Barr)
    Rp = U @ Vt
    if np.linalg.det(Rp) < 0:
        Vt = Vt.copy(); Vt[-1, :] *= -1.0; Rp = U @ Vt
    rv    = _Rot.from_matrix(Rp)
    vec   = rv.as_rotvec(degrees=True)
    total = float(np.linalg.norm(vec))
    axis  = vec / (total + 1e-12)
    eul   = rv.as_euler("XYZ", degrees=True)
    K, _  = _rq3(Barr)
    print(f"    volume scale (det)   : {detB:.4f}x   (>1 = subject was ENLARGED)")
    print(f"    principal stretches  : {S[0]:.4f} / {S[1]:.4f} / {S[2]:.4f}"
          f"   anisotropy {S[0]/max(S[2],1e-9):.4f}")
    print(f"    TOTAL ROTATION       : {total:.3f} deg")
    print(f"      about array direction ({axis[0]:+.3f}, {axis[1]:+.3f}, {axis[2]:+.3f})")
    print(f"    euler about axes 0/1/2 (XYZ): {eul[0]:+.3f} / {eul[1]:+.3f} / {eul[2]:+.3f} deg")
    print(f"    per-axis scale (RQ)  : {K[0,0]:.4f} / {K[1,1]:.4f} / {K[2,2]:.4f}")
    print(f"    shear (RQ off-diag)  : {K[0,1]:+.4f}  {K[0,2]:+.4f}  {K[1,2]:+.4f}")
    ctr_s = np.asarray(subj.origin, float) + D_s @ (sp_s * (np.array(subj.shape, float) - 1) / 2.0)
    ctr_t = B @ (ctr_s - c - t) + c
    ctr_0 = np.asarray(tpl.origin, float) + D_t @ (sp_t * (np.array(tpl.shape, float) - 1) / 2.0)
    d_mm  = Dt_i @ (ctr_t - ctr_0)
    print(f"    subject centre lands {np.linalg.norm(d_mm):.2f} mm from template centre: "
          f"({d_mm[0]:+.2f}, {d_mm[1]:+.2f}, {d_mm[2]:+.2f}) mm on axes 0/1/2")

    # 2.2.6.2  per-axis voxel step
    print("  --- ONE SUBJECT VOXEL STEP, MEASURED IN TEMPLATE SPACE ---")
    k_thick, info = int(np.argmax(sp_s)), {}
    for k in range(3):
        step = B @ (D_s[:, k] * sp_s[k])
        L    = float(np.linalg.norm(step))
        u    = step / (L + 1e-12)
        comp = np.abs(Dt_i @ u)
        j    = int(np.argmax(comp))
        tilt = float(np.degrees(np.arccos(min(1.0, float(comp[j])))))
        mark = "  <== THICK AXIS" if k == k_thick else ""
        print(f"    subj axis {k} ({sp_s[k]:.4f} mm) -> {L:.4f} mm, nearest tpl axis {j}, "
              f"tilt {tilt:.2f} deg, = {L/float(sp_t[j]):.3f} tpl voxels{mark}")
        if k == k_thick:
            info = {"subj_axis": k, "tpl_axis": j, "len_mm": L,
                    "factor": L / float(sp_t[j]), "tilt": tilt}

    # 2.2.6.3  occupancy extents
    print("  --- OCCUPANCY (data vs background) ---")
    e_tpl = _extent(TEMPLATE_MRI_SQ)
    e_nat = _extent(scans[sid])
    e_wrp = _extent(regs[sid].get("warped_template_grid", regs[sid]["warped"]))
    _print_extent("template ", e_tpl)
    _print_extent("native   ", e_nat)
    _print_extent("warped   ", e_wrp)
    # 2.2.6.4  empirical volume scale vs the affine determinant
    emp_vol = e_wrp["vol_mm3"] / max(e_nat["vol_mm3"], 1e-9)
    print(f"    empirical volume scale (warped brain / native brain) = {emp_vol:.4f}x")
    print(f"    affine determinant                                   = {detB:.4f}x")
    print("    (a few percent apart = interpolation halo; a big gap = something is wrong)")

    jt   = info["tpl_axis"]
    nacq = e_nat["rows"][info["subj_axis"]]["n"]
    nwrp = e_wrp["rows"][jt]["n"]
    print(f"    thick axis: {nacq} acquired slices with data -> {nwrp} template slices "
          f"with data = {nwrp/max(nacq,1):.3f} per acquired slice")
    print(f"    geometric factor said {info['factor']:.3f} -- these two MUST agree")
    half  = 0.5 * max(e_wrp["rows"][a]["mm"] for a in (0, 1, 2) if a != jt)
    # 2.2.6.5  obliquity
    smear = 2.0 * half * np.sin(np.radians(info["tilt"])) / float(sp_t[jt])
    print(f"    OBLIQUITY: tilt {info['tilt']:.2f} deg across {2*half:.1f} mm spreads one "
          f"acquired plane over ~{smear:.1f} template slices")
    if smear <= info["factor"]:
        print("      -> acquired planes are near-parallel to template coronal slices.")
    else:
        print("      -> OBLIQUE: no single template coronal slice equals one acquired slice.")
    return info

_regs  = load_state("regs")
_scans = {sid: d["brain"] for sid, d in load_state("preproc").items()}
print("=== TRANSFORM REPORT ===")
_all = {sid: transform_report(sid, _regs, _scans) for sid in _regs}
print("\n=== SUMMARY ===")
for _sid, _i in _all.items():
    print(f"  {_sid}: thick subj axis {_i['subj_axis']} -> tpl axis {_i['tpl_axis']}, "
          f"{_i['factor']:.3f} tpl voxels per acquired slice, tilt {_i['tilt']:.2f} deg")


## Stage 3 - Parcellation and initial export

Boundaries are traced from the template labels into a node/element graph in which every
element carries the two region codes it separates, so a shared border is a single line.
The graph is built once per slice and handed to every subject identically.

In [ ]:
# =====================================================================================
# 3.1  PARCELLATION: trace every slice into a boundary graph
#
# 3.1.1  apply the hemisphere mask
# 3.1.2  fragment relabelling (before the voxel threshold, so arcs and seeds agree)
# 3.1.3  trace boundaries into a graph
# 3.1.4  smoothing (Gaussian low-pass) and simplification (Douglas-Peucker)
# 3.1.5  separate closed rings from open arcs; 3.1.6 compute seeds
# 3.1.7  the template graph is built once per slice and copied to every subject
# 3.1.9  the trace is cached under a fingerprint of the labels plus the trace parameters
# =====================================================================================
import numpy as np, nibabel as nib
from tqdm.auto import tqdm

# ----------------------------- TOGGLES -----------------------------------------------
# Boundary smoothing. Tracing a label image gives an axis-aligned pixel staircase; a
# low-pass filter on the traced coordinates runs BEFORE the single simplify, so Douglas-
# Peucker drops redundant points along a smooth curve instead of re-finding step corners.
# vertices by deviation; this only exists for Illustrator ergonomics and is LOSSY.
# half-lines, one per region, collapsed back to one on import.

# 3.1.1  apply the hemisphere mask (zeroes the right hemisphere)
def _hemisphere_mask(label_slice, hemisphere):
    """Zero the half not kept; axis 0 of the raw slice is L-R."""
    if hemisphere == "whole":
        return label_slice, None
    H_lr = label_slice.shape[0]
    mid = H_lr / 2.0
    out = label_slice.copy()
    rows = np.arange(H_lr)
    if hemisphere == "left":
        out[rows >= mid, :] = 0
    elif hemisphere == "right":
        out[rows < mid, :] = 0
    else:
        raise ValueError(f"HEMISPHERE must be 'whole'|'left'|'right', got {hemisphere!r}")
    return out, mid

# 3.1.4.1  Gaussian low-pass filter on the traced boundary
def _smooth_ring(coords, sigma):
    """Gaussian low-pass on a traced polyline, before simplification."""
    from scipy.ndimage import gaussian_filter1d
    pts = np.asarray(coords, float)
    if sigma <= 0 or len(pts) < 5:
        return pts
    if np.allclose(pts[0], pts[-1]):
        ring = pts[:-1]
        x = gaussian_filter1d(ring[:, 0], sigma, mode="wrap")
        y = gaussian_filter1d(ring[:, 1], sigma, mode="wrap")
        out = np.column_stack([x, y])
        return np.vstack([out, out[0]])
    x = gaussian_filter1d(pts[:, 0], sigma, mode="nearest")
    y = gaussian_filter1d(pts[:, 1], sigma, mode="nearest")
    out = np.column_stack([x, y])
    out[0], out[-1] = pts[0], pts[-1]
    return out

# 3.1.4.3  handle budget: raise the DP tolerance until handles <= TRACE_MAX_HANDLES
def _simplify_to_budget(coords, tol_px, max_pts=None):
    """Douglas-Peucker to at most max_pts vertices. Bisects for the smallest tolerance that
    fits the budget. Returns (pts, achieved_tol, deviation_px)."""
    from shapely.geometry import LineString
    P = np.asarray(coords, float)
    if len(P) < 3:
        return P, tol_px, 0.0
    closed = np.allclose(P[0], P[-1])
    if max_pts is None or len(P) <= max_pts:
        return P, tol_px, 0.0
    lo, hi = tol_px, max(tol_px * 2, 1.0)
    L = LineString([tuple(p) for p in P])
    for _ in range(40):
        if len(np.array(L.simplify(hi).coords)) <= max_pts:
            break
        hi *= 1.6
    for _ in range(24):
        mid = 0.5 * (lo + hi)
        if len(np.array(L.simplify(mid).coords)) <= max_pts:
            hi = mid
        else:
            lo = mid
    Q = np.array(L.simplify(hi).coords)
    if closed and not np.allclose(Q[0], Q[-1]):
        Q = np.vstack([Q, Q[0]])
    return Q, hi, float(ef.max_deviation(P, Q))

# 3.1.6 / 3.1.6.1  seed = distance-transform maximum of the region's voxel mask
def _seed_points_on_slice(label_slice, min_region_voxels=MIN_REGION_VOXELS):
    """One deep interior point per region: the distance-transform maximum."""
    from scipy.ndimage import distance_transform_edt
    seeds = []
    for rid in np.unique(label_slice):
        if rid == 0:
            continue
        m = label_slice == rid
        if m.sum() < min_region_voxels:
            continue
        dt = distance_transform_edt(m)
        iy, ix = np.unravel_index(np.argmax(dt), dt.shape)
        seeds.append(((float(ix), float(iy)), int(rid)))
    return seeds

# 3.1.3  trace boundaries into a graph
def label_slice_to_graph(label_slice, min_region_voxels=MIN_REGION_VOXELS,
                         simplify_tol=SIMPLIFY_TOL, adaptive=True):
    """Trace one integer label slice into a node/element graph where each element carries the
    TWO region codes it separates, so a shared border is a SINGLE line."""
    from collections import defaultdict
    from shapely.geometry import LineString
    from shapely.ops import linemerge

    # one unique code per connected piece, BEFORE the voxel threshold, so the arcs and the
    # seeds returned at the end share one fragment decomposition [edge_fusion 0b]
    # 3.1.2 / 3.1.2.2  fragment relabelling, BEFORE the voxel threshold
    lab = ef.fragment_relabel_slice(label_slice)
    H, W = lab.shape
    # 3.1.3.1 / 3.1.2.3  drop fragments below MIN_REGION_VOXELS
    keep = np.zeros_like(lab)                       # drop regions below the voxel threshold
    for rid in np.unique(lab):
        if rid == 0:
            continue
        m = lab == rid
        if m.sum() >= min_region_voxels:
            keep[m] = rid
    lab = keep

    # 3.1.3.2  walk the slice and record where voxels change region
    segs = defaultdict(list)                        # walk the slice, record region changes
    for c in range(W - 1):
        diff = lab[:, c] != lab[:, c + 1]
        for r in np.where(diff)[0]:
            a, b = sorted((int(lab[r, c]), int(lab[r, c + 1])))
            # 3.1.3.3  each segment records the two regions it separates
            segs[(a, b)].append([(c + 0.5, r - 0.5), (c + 0.5, r + 0.5)])
    for r in range(H - 1):
        diff = lab[r, :] != lab[r + 1, :]
        for c in np.where(diff)[0]:
            a, b = sorted((int(lab[r, c]), int(lab[r + 1, c])))
            segs[(a, b)].append([(c - 0.5, r + 0.5), (c + 0.5, r + 0.5)])

    g = bg.BoundaryGraph()
    max_dev = 0.0
    for (a, b), seglist in segs.items():            # merge same-code segments into polylines
        # 3.1.3.4  merge same-code segments into polylines
        merged = linemerge([LineString(s) for s in seglist])
        geoms = merged.geoms if merged.geom_type == "MultiLineString" else [merged]
        for geom in geoms:
            if SMOOTH_BOUNDARIES:
                pts = _smooth_ring(np.array(geom.coords), SMOOTH_SIGMA)
                if simplify_tol > 0 and len(pts) >= 2:
                    # 3.1.4.2  Douglas-Peucker simplification
                    pts = np.array(LineString(pts).simplify(simplify_tol).coords)
            else:
                pts = np.array(geom.simplify(simplify_tol).coords)
            # 3.1.4.3  handle budget
            if adaptive and len(pts) >= 3 and TRACE_MAX_HANDLES:
                pts, _t, dev = _simplify_to_budget(pts, simplify_tol, TRACE_MAX_HANDLES)
                max_dev = max(max_dev, dev)
            if len(pts) < 2:
                continue
            prev = None
            for xy in pts:
                ni = g.add_node(xy)
                if prev is not None:
                    g.add_element(prev, ni, a, b)   # both codes set at creation
                prev = ni
    g.finalize_nodes()          # add_node defers .nodes; materialise once after the loop

    if max_dev > 0:
        print(f"  !! TRACE_MAX_HANDLES={TRACE_MAX_HANDLES} cost up to {max_dev:.3f}px of "
              f"boundary deviation (lossy, and applied BEFORE the expert QA).")
    return g, _seed_points_on_slice(lab, min_region_voxels)

# 3.1.9.1  cache key: label volume plus the trace parameters
def _template_fingerprint(lab, min_region_voxels, simplify_tol):
    """Cache key: reject a trace built against a different label volume or trace params."""
    ids = np.unique(lab); ids = ids[ids > 0]
    return {"labels_path": TEMPLATE_LABELS,
            "shape": tuple(int(s) for s in lab.shape),
            "n_ids": int(ids.size), "id_checksum": int(ids.sum()),
            "hemisphere": HEMISPHERE, "slice_axis": SLICE_AXIS,
            "smooth_boundaries": bool(SMOOTH_BOUNDARIES), "smooth_sigma": float(SMOOTH_SIGMA),
            "trace_max_handles": TRACE_MAX_HANDLES,
            "min_region_voxels": int(min_region_voxels),
            "simplify_tol": float(simplify_tol)}

def view_template_slice(slice_index, save=None):
    """DIAGNOSTIC: render one slice of the label volume to confirm SLICE_AXIS and coverage."""
    import matplotlib.pyplot as plt
    lab = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    sl = np.take(lab, slice_index, axis=SLICE_AXIS)
    print(f"slice {slice_index}: shape {sl.shape}, {len(np.unique(sl))-1} regions present")
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(np.rot90(sl), cmap="nipy_spectral"); ax.set_title(f"labels, slice {slice_index}")
    ax.axis("off")
    if save:
        fig.savefig(save, dpi=150, bbox_inches="tight")
    return slice_index

# 3.1.7  trace once per slice, then copy to every subject
def propagate_all_slices(min_region_voxels=MIN_REGION_VOXELS, simplify_tol=SIMPLIFY_TOL,
                         slice_subset=SLICE_SUBSET, force=False):
    """Trace every non-empty coronal slice, then write one ASCII line file per subject per
    slice. Empty slices are skipped; the trace is cached under a fingerprint."""
    regs = load_state("regs")
    lab  = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    n_slices = lab.shape[SLICE_AXIS]
    fp = _template_fingerprint(lab, min_region_voxels, simplify_tol)

    cached    = load_state("template_graphs", default=None)
    cached_fp = load_state("template_fingerprint", default=None)
    # 3.1.9  trace cache accepted only when the fingerprint matches
    if cached and cached_fp == fp and not force:
        template_graphs = cached
        midline_lr = load_state("trace_meta", default={}).get("midline_lr")
        print(f"CACHE HIT: {len(template_graphs)} traced slices reused.")
    else:
        if cached is not None and cached_fp != fp:
            print("  !! cache REJECTED -- template or trace params changed.")
            for k in fp:
                if (cached_fp or {}).get(k) != fp[k]:
                    print(f"       {k}: cached={(cached_fp or {}).get(k)!r}  now={fp[k]!r}")
        lo, hi = (0, n_slices) if slice_subset is None else slice_subset
        template_graphs, midline_lr = {}, None
        t0 = time.time()
        for sidx in tqdm(range(lo, min(hi, n_slices)), desc="trace", unit="slice"):
            sl_raw = np.take(lab, sidx, axis=SLICE_AXIS)
            if (sl_raw > 0).sum() < min_region_voxels:
                continue
            sl, mid = _hemisphere_mask(sl_raw, HEMISPHERE)
            if (sl > 0).sum() < min_region_voxels:
                continue
            midline_lr = mid if mid is not None else midline_lr
            template_graphs[sidx] = label_slice_to_graph(sl, min_region_voxels,
                                                         simplify_tol, adaptive=True)
        save_state("template_graphs", template_graphs)
        save_state("template_fingerprint", fp)
        print(f"Traced {len(template_graphs)} non-empty slices of {n_slices} "
              f"in {time.time()-t0:.1f}s.")

    if not template_graphs:
        raise RuntimeError("no non-empty slices traced -- check TEMPLATE_LABELS and SLICE_AXIS")

    out = {}                                    # identical copy to every subject
    # 3.1.7  the template graph is applied identically to every subject
    for sid in tqdm(list(regs), desc="seed subjects", unit="subj"):
        subj = {}
        for sidx, (g, seeds) in template_graphs.items():
            gc = bg.BoundaryGraph(nodes=np.array(g.nodes).copy(),
                                  elements=[e[:] for e in g.elements])
            outp = f"{RUN_DIR}/lines_propagated/{sid}_slice_{sidx:03d}.txt"
            # 3.1.8  written as an ASCII line file
            bg.write_ascii(gc, outp); subj[sidx] = outp
        out[sid] = subj
    save_state("propagated", out)

    slices = sorted(template_graphs)
    rep    = slices[len(slices) // 2]           # representative slice for previews / QA
    save_state("trace_meta", {"slice_index": rep, "slices": slices,
                              "label_detail": LABEL_DETAIL, "line_mode": LINE_MODE,
                              "double_offset": LINE_DOUBLE_OFFSET,
                              "trace_max_handles": TRACE_MAX_HANDLES,
                              "simplify_tol": simplify_tol,
                              "hemisphere": HEMISPHERE, "midline_lr": midline_lr})

    npts = [len(a.pts) for g, _ in template_graphs.values() for a in ef.graph_to_arcs(g)]
    nodes_tot = sum(len(g.nodes) for g, _ in template_graphs.values())
    checks = [
        ("slices traced", len(template_graphs) > 0, "check TEMPLATE_LABELS / SLICE_AXIS"),
        ("every subject seeded", len(out) == len(regs), "a subject is missing from regs"),
        ("graph coords lie inside the label grid",
         all(np.asarray(g.nodes)[:, 0].max() <= lab.shape[2] and
             np.asarray(g.nodes)[:, 1].max() <= lab.shape[0]
             for g, _ in template_graphs.values()),
         "stale trace: coords exceed the grid -> re-run with force=True"),
        ("handles per boundary are draggable in Illustrator",
         (max(npts) <= 200 if npts else True),
         f"max {max(npts) if npts else 0} handles on one boundary -> raise simplify_tol"),
    ]
    report_outputs("CELL 6 propagate (all slices)",
                   state_keys=["template_graphs", "propagated", "trace_meta",
                               "template_fingerprint"], checks=checks)
    print(f"\nPropagated {len(template_graphs)} slices x {len(out)} subjects "
          f"(hemisphere='{HEMISPHERE}', {nodes_tot} nodes total).")
    print(f"  slices {slices[0]}..{slices[-1]}, representative = {rep}")
    if npts:
        print(f"  Douglas-Peucker anchors (simplify_tol={simplify_tol}px): "
              f"{min(npts)}..{max(npts)} pts/arc, median {int(np.median(npts))}")
    if HEMISPHERE != "whole":
        print(f"  single hemisphere kept (midline L-R row = {midline_lr:.1f}); "
              f"CELL 10 mirrors it back after fusion.")
    return out

propagate_all_slices()


In [ ]:
# =====================================================================================
# 3.2 / 4.4  SVG ROUND-TRIP MACHINERY   (writer + reader; no driver)
#
# 3.2.3    writer: background (locked), boundaries (editable), labels
# 3.2.3.4  double-line mode writes each border twice, one path per owning region
# 4.4.1    reader: codes recovered from id="bnd_A_B_k" (Illustrator strips data-*)
# 4.4.2    _xHH_ escapes undone; 4.4.3 ancestor transforms composed
# 4.4.4    Bezier segments flattened at flatten_px
# 4.4.5    closed uncoded path -> new expert id (9001+); open uncoded path counted only
# 4.4.6    double-line pairs collapsed back to one line
# =====================================================================================
import base64, io, os, csv, colorsys, math
import xml.etree.ElementTree as ET
import re as _re
from xml.sax.saxutils import escape as _xml_escape
import numpy as np, nibabel as nib

# ---- display / labelling options -----------------------------------------------------
# sigma above kp_alpha would erase the notches the key-point machinery exists to preserve.

def _fmt(v):
    """Coordinate formatter. Quantisation happens on write only; grid_write=None -> float."""
    p = FUSION_PARAMS
    if p.grid_write:
        v = round(float(v) / p.grid_write) * p.grid_write
    return f"{float(v):.{p.svg_decimals}f}"

def _number_regions(anchors):
    rids = sorted({a[0] for a in anchors})
    return rids, {rid: i + 1 for i, rid in enumerate(rids)}

def region_color(rid):
    """Golden-ratio hue per id; magenta reserved for UNLABELED (QA gap) ids.
    Fragments of one region share the BASE id's colour."""
    rid = ef.fragment_base(int(rid))
    p = FUSION_PARAMS
    if p.unlabeled_id_start <= rid < p.new_id_start:
        return (255, 0, 255)
    h = (rid * 0.61803398875) % 1.0
    r, g, b = colorsys.hsv_to_rgb(h, 0.65, 0.92)
    return (round(r*255), round(g*255), round(b*255))

def _rgb_hex(rgb):
    return "#{:02x}{:02x}{:02x}".format(*rgb)

def load_region_table(lut_csv=None):
    """{id: {abbrev, name, rgb}} from the LUT CSV, for SVG labels and colours."""
    lut_csv = lut_csv or TEMPLATE_LUT
    table = {}
    if not (lut_csv and os.path.exists(lut_csv)):
        print(f"  !! LUT not found at {lut_csv} -- labels fall back to numeric ids.")
        return table
    rows = list(csv.DictReader(open(lut_csv, newline="")))
    if not rows:
        print(f"  !! LUT {lut_csv} is empty."); return table
    cols = {c.lower().strip(): c for c in rows[0].keys()}
    id_col   = cols.get("id")
    abbr_col = cols.get("abbreviation") or cols.get("abbrev") or cols.get("label")
    name_col = cols.get("name") or cols.get("long_name") or cols.get("region")
    hex_col  = cols.get("color_hex") or cols.get("hex")
    for row in rows:
        try: rid = int(float(row.get(id_col)))
        except (TypeError, ValueError): continue
        abbrev = (row.get(abbr_col) if abbr_col else None) or str(rid)
        name   = (row.get(name_col) if name_col else None) or abbrev
        abbrev = str(abbrev).split(":")[-1].strip(); name = str(name).strip()
        rgb = region_color(rid)
        if hex_col and row.get(hex_col, "").startswith("#"):
            h = row[hex_col].lstrip("#")
            try: rgb = (int(h[0:2],16), int(h[2:4],16), int(h[4:6],16))
            except ValueError: pass
        table[rid] = {"abbrev": abbrev, "name": name, "rgb": rgb}
    return table

def _abbrev_for(table, rid):
    code = int(rid)
    rid, fx = ef.fragment_base(code), ef.fragment_index(code)   # name comes from the BASE
    p = FUSION_PARAMS
    if rid == 0: return "bg"
    if p.unlabeled_id_start <= rid < p.new_id_start: ab = f"UNLAB{rid}"
    elif rid >= p.new_id_start: ab = f"NEW{rid}"
    else: ab = table.get(rid, {}).get("abbrev", str(rid))
    return ab if fx == 0 else f"{ab}#{fx}"

# ---- orientation: voxel frame <-> display frame (coronal) ----------------------------
def _orient_image(sl):
    img = sl.T
    img = img[::-1, :]
    if FLIP_LR: img = img[:, ::-1]
    return img

def _orient_xy(x_col, y_row, H_lr, W_si):
    nr = (W_si - 1) - x_col
    nc = (H_lr - 1 - y_row) if FLIP_LR else y_row
    return nc, nr

def _orient_arr(P, H_lr, W_si):
    P = np.asarray(P, float)
    nr = (W_si - 1) - P[:, 0]
    nc = ((H_lr - 1) - P[:, 1]) if FLIP_LR else P[:, 1]
    return np.column_stack([nc, nr])

def _unorient_arr(D, H_lr, W_si):
    D = np.asarray(D, float)
    nc = ((H_lr - 1) - D[:, 0]) if FLIP_LR else D[:, 0]
    x_col = (W_si - 1) - D[:, 1]
    return np.column_stack([x_col, nc])

def _slice_to_png_bytes(t1_volume, slice_index, axis, hemisphere="whole", midline_lr=None):
    """T1 slice -> a PNG the SVG embeds as the locked underlay."""
    import matplotlib; matplotlib.use("Agg")
    import matplotlib.pyplot as plt
    raw = np.take(t1_volume, slice_index, axis=axis).astype(np.float32)
    img = _orient_image(raw)
    lo, hi = np.percentile(img[img > 0], (1, 99)) if (img > 0).any() else (0, 1)
    img = np.clip((img - lo) / (hi - lo + 1e-9), 0, 1)
    if hemisphere != "whole" and midline_lr is not None:
        W_img = img.shape[1]
        mid_col = int(round(midline_lr))
        keep_low = (hemisphere == "left")
        if FLIP_LR:
            keep_low = not keep_low
            mid_col = W_img - mid_col
        if keep_low: img[:, mid_col:] = 0.0
        else:        img[:, :mid_col] = 0.0
    buf = io.BytesIO(); h, w = img.shape
    fig = plt.figure(figsize=(w/100, h/100), dpi=100); ax = fig.add_axes([0, 0, 1, 1])
    ax.imshow(img, cmap="gray", origin="upper"); ax.axis("off")
    fig.savefig(buf, format="png", dpi=100); plt.close(fig)
    return buf.getvalue(), w, h, raw.shape

def _label_points_on_slice(label_slice, label_all_instances=None, min_component_voxels=None):
    """One label anchor per connected component, at its distance-transform maximum."""
    from scipy.ndimage import distance_transform_edt, label as cc_label
    if label_all_instances is None:  label_all_instances = LABEL_ALL_INSTANCES
    if min_component_voxels is None: min_component_voxels = MIN_COMPONENT_VOXELS
    label_slice = ef.fragment_relabel_slice(label_slice, min_voxels=1)
    anchors = []
    for rid in np.unique(label_slice):
        if rid == 0: continue
        comps, n = cc_label(label_slice == rid)
        allc = []
        for ci in range(1, n + 1):
            comp = comps == ci
            dt = distance_transform_edt(comp)
            iy, ix = np.unravel_index(np.argmax(dt), dt.shape)
            allc.append((int(rid), float(ix), float(iy), float(dt[iy, ix]), int(comp.sum())))
        if not allc: continue
        kept = [c for c in allc if c[4] >= min_component_voxels]
        if not kept: kept = [max(allc, key=lambda t: t[4])]
        ninst = len(kept)
        if label_all_instances:
            anchors.extend([(r, x, y, c, ninst) for (r, x, y, c, _s) in kept])
        else:
            r, x, y, c, _s = max(kept, key=lambda t: t[3])
            anchors.append((r, x, y, c, ninst))
    if not FUSION_PARAMS.label_all_fragments:
        anchors = ef.filter_anchors_largest_fragment(anchors)
    return anchors

def _place_labels(anchors, label_text_of, H_lr, W_si):
    """Font size from the local clearance, so text fits inside its region."""
    from collections import defaultdict
    by_region = defaultdict(list)
    for rid, x, y, clearance, _n in anchors:
        by_region[rid].append((x, y, clearance))
    labels = []
    for rid, comps in by_region.items():
        txt = str(label_text_of.get(rid, rid)); ndig = max(len(txt), 1)
        for (x, y, clearance) in comps:
            d = 2.0 * clearance
            fs = min((d * 0.9) / (ndig * LABEL_FIT), d * LABEL_HEIGHT_FRAC)
            fs = min(max(fs, LABEL_FS_MIN), LABEL_FS_MAX)
            xd, yd = _orient_xy(x, y, H_lr, W_si)
            labels.append({"rid": rid, "cx": xd, "cy": yd, "fs": fs, "txt": txt})
    return labels

def _smooth_contour(pts, closed, sigma):
    """Gaussian smoothing of a display polyline. Endpoints pinned; interior vertices move."""
    from scipy.ndimage import gaussian_filter1d
    P = np.asarray(pts, float)
    if len(P) < 4 or sigma <= 0: return P
    if closed and np.allclose(P[0], P[-1]): P = P[:-1]
    ring = np.vstack([P, P[0]]) if closed else P
    seg = np.sqrt((np.diff(ring, axis=0) ** 2).sum(1)); d = np.r_[0, np.cumsum(seg)]
    if d[-1] <= 0: return P
    u = np.arange(0, d[-1], 1.0)
    xs = np.interp(u, d, ring[:, 0]); ys = np.interp(u, d, ring[:, 1])
    if closed:
        xs = gaussian_filter1d(xs, sigma, mode="wrap"); ys = gaussian_filter1d(ys, sigma, mode="wrap")
        out = np.c_[xs, ys]; return np.vstack([out, out[0]])
    end0, end1 = ring[0].copy(), ring[-1].copy()
    xs = gaussian_filter1d(xs, sigma, mode="nearest"); ys = gaussian_filter1d(ys, sigma, mode="nearest")
    out = np.c_[xs, ys]; out[0], out[-1] = end0, end1
    return out

def _offset_polyline(disp, offset):
    """Offset a display polyline along its per-vertex normal (double line mode)."""
    P = np.asarray(disp, float)
    if len(P) < 2: return P
    closed = np.allclose(P[0], P[-1])
    Q = P[:-1] if closed else P
    n = len(Q)
    tang = np.zeros_like(Q)
    for i in range(n):
        a = Q[i-1] if (closed or i > 0) else Q[i]
        b = Q[(i+1) % n] if (closed or i < n-1) else Q[i]
        t = b - a; nrm = np.hypot(*t)
        tang[i] = t / nrm if nrm > 1e-9 else np.array([1.0, 0.0])
    normal = np.c_[-tang[:, 1], tang[:, 0]]
    out = Q + offset * normal
    return np.vstack([out, out[0]]) if closed else out

# WRITER
# 3.2.3  write the layered SVG
def graph_to_svg(g, path, t1_volume=None, slice_index=None, axis=1,
                 label_anchors=None, table=None, stroke=1.2,
                 line_mode="single", double_offset=0.35, label_detail="abbrev",
                 hemisphere="whole", midline_lr=None, smooth_sigma=None,
                 regions=None, fill_opacity=0.0):
    """Write one slice to a layered SVG."""
    if not len(g.nodes) or t1_volume is None or slice_index is None:
        open(path, "w").write("<svg xmlns='http://www.w3.org/2000/svg'/>"); return
    table = table or {}
    sigma = EXPORT_SMOOTH_SIGMA if smooth_sigma is None else smooth_sigma
    png, W, H, (H_lr, W_si) = _slice_to_png_bytes(
        t1_volume, slice_index, axis, hemisphere=hemisphere, midline_lr=midline_lr)
    b64 = base64.b64encode(png).decode("ascii")

    show_text = (label_detail != "none")
    region_ids, _num = _number_regions(label_anchors or [])
    abbrev_of = {rid: _abbrev_for(table, rid) for rid in region_ids}
    labels = _place_labels(label_anchors, abbrev_of, H_lr, W_si) \
             if (label_anchors and show_text) else []

    out = ['<?xml version="1.0" encoding="UTF-8"?>',
           f'<svg xmlns="http://www.w3.org/2000/svg" xmlns:xlink="http://www.w3.org/1999/xlink" '
           f'xmlns:inkscape="http://www.inkscape.org/namespaces/inkscape" '
           f'xmlns:sodipodi="http://sodipodi.sourceforge.net/DTD/sodipodi-0.0.dtd" '
           f'viewBox="0 0 {W:.1f} {H:.1f}" width="{W:.1f}" height="{H:.1f}">']

    # 3.2.3.1  background (white page + T1 raster)
    # -- LAYER 1: background (page + T1 raster), locked --------------------------------
    out.append('<g inkscape:groupmode="layer" inkscape:label="background" id="layer_background" '
               'sodipodi:insensitive="true" style="pointer-events:none">')
    out.append(f'<rect x="0" y="0" width="{W:.1f}" height="{H:.1f}" fill="white"/>')
    out.append(f'<image x="0" y="0" width="{W}" height="{H}" '
               f'xlink:href="data:image/png;base64,{b64}" style="pointer-events:none"/>')
    out.append('</g>')

    # -- LAYER 1b: the polygon atlas (optional), locked --------------------------------
    if regions and fill_opacity > 0:
        out.append('<g inkscape:groupmode="layer" inkscape:label="atlas_faces" id="layer_faces" '
                   f'sodipodi:insensitive="true" style="pointer-events:none" '
                   f'opacity="{fill_opacity:.2f}">')
        for rid, faces in sorted(regions.items()):
            col = _rgb_hex(region_color(rid))
            for k, poly in enumerate(faces):
                rings = [np.asarray(poly.exterior.coords, float)] + \
                        [np.asarray(r.coords, float) for r in poly.interiors]
                d = []
                for R in rings:                       # even-odd fill -> holes are real holes
                    D = _orient_arr(R, H_lr, W_si)
                    d.append("M " + " L ".join(f"{_fmt(x)},{_fmt(y)}" for x, y in D) + " Z")
                out.append(f'<path d="{" ".join(d)}" fill="{col}" fill-rule="evenodd" '
                           f'stroke="none" id="face_{int(rid)}_{k}" data-region="{int(rid)}"/>')
        out.append('</g>')

    # 3.2.3.2  boundaries, one polyline per arc
    # -- LAYER 2: boundaries, one polyline per ARC, editable ---------------------------
    out.append('<g inkscape:groupmode="layer" inkscape:label="boundaries" id="layer_boundaries" '
               'fill="none" stroke-linejoin="round" stroke-linecap="round">')
    for k, arc in enumerate(ef.graph_to_arcs(g)):
        mA, mB = int(arc.code[0]), int(arc.code[1])
        P = np.vstack([arc.pts, arc.pts[0]]) if arc.closed else arc.pts
        disp = _orient_arr(P, H_lr, W_si)
        if len(disp) < 2:
            continue
        if sigma > 0:
            disp = _smooth_contour(disp, bool(arc.closed), sigma)
        na, nb = _abbrev_for(table, mA), _abbrev_for(table, mB)
        title  = f'<title>{_xml_escape(str(na))} | {_xml_escape(str(nb))}</title>'
        eid = f"bnd_{mA}_{mB}_{k}"
        if line_mode == "double":
            half = stroke / 2.0
            colA = _rgb_hex(region_color(mA)) if mA > 0 else "#888888"
            colB = _rgb_hex(region_color(mB)) if mB > 0 else "#888888"
            out.append(f'<g id="arc_{eid}" data-arc="{k}">')
            # 3.2.3.4  double-line mode: two coincident half-lines, one per owning region
            for si, (sgn, col, mat) in enumerate(((+1.0, colA, mA), (-1.0, colB, mB))):
                od = _offset_polyline(disp, sgn * double_offset)
                pts_str = " ".join(f"{_fmt(x)},{_fmt(y)}" for (x, y) in od)
                out.append(f'<polyline id="{eid}_s{si}" points="{pts_str}" stroke="{col}" '
                           f'stroke-width="{half:.3f}" fill="none" data-matA="{mA}" '
                           f'data-matB="{mB}" data-side="{mat}" data-arc="{k}">'
                           + title + '</polyline>')
            out.append('</g>')
        else:
            rid = mA if mA > 0 else mB
            col = _rgb_hex(region_color(rid)) if rid > 0 else "#888888"
            pts_str = " ".join(f"{_fmt(x)},{_fmt(y)}" for (x, y) in disp)
            out.append(f'<polyline id="{eid}" points="{pts_str}" stroke="{col}" '
                       f'stroke-width="{stroke}" fill="none" data-matA="{mA}" '
                       f'data-matB="{mB}" data-arc="{k}">' + title + '</polyline>')
    out.append('</g>')

    # 3.2.3.3  labels
    # -- LAYER 3: labels ---------------------------------------------------------------
    if labels:
        out.append('<g inkscape:groupmode="layer" inkscape:label="labels" id="layer_labels" '
                   'font-family="sans-serif" font-weight="bold" '
                   'text-anchor="middle" dominant-baseline="central">')
        _seen = {}
        for L in labels:
            col = _rgb_hex(region_color(L["rid"]))
            pos = f'x="{L["cx"]:.2f}" y="{L["cy"]:.2f}" font-size="{L["fs"]:.2f}"'
            _n = _seen.get(L["rid"], 0); _seen[L["rid"]] = _n + 1
            txt = _xml_escape(str(L["txt"]))
            out.append(f'<g id="label_{int(L["rid"])}_{_n}" data-region="{L["rid"]}">')
            out.append(f'<text {pos} fill="black" stroke="black" '
                       f'stroke-width="{L["fs"]*LABEL_OUTLINE_FRAC:.3f}" '
                       f'stroke-linejoin="round">{txt}</text>')
            out.append(f'<text {pos} fill="{col}" data-region="{L["rid"]}">{txt}</text>')
            out.append('</g>')
        out.append('</g>')

    out.append('</svg>')
    doc = "\n".join(out)
    try:
        ET.fromstring(doc)
    except ET.ParseError as e:
        ln = getattr(e, "position", (0, 0))[0]
        lines = doc.split("\n")
        print(f"[svg-check] {path} is malformed: {e}")
        for j in range(max(0, ln - 3), min(len(lines), ln + 2)):
            print(f"[svg-check]   {j+1}: {lines[j]}")
    open(path, "w").write(doc)

# READER
# 4.4.2  un-escape _xHH_ sequences in Illustrator ids
def _svg_unescape(s):
    """Illustrator writes _x5F_ for '_', _x2D_ for '-', etc."""
    return _re.sub(r"_x([0-9A-Fa-f]{2})_", lambda m: chr(int(m.group(1), 16)), s or "")

def _mm(A, B):
    A3 = np.vstack([A, [0., 0., 1.]]); B3 = np.vstack([B, [0., 0., 1.]])
    return (A3 @ B3)[:2]

# 4.4.3  parse one SVG transform attribute
def _parse_transform(t):
    """Compose an SVG transform string into a 2x3 affine [[a,c,e],[b,d,f]]."""
    M = np.array([[1., 0., 0.], [0., 1., 0.]])
    if not t:
        return M
    for name, args in _re.findall(r"(\w+)\s*\(([^)]*)\)", t):
        v = [float(x) for x in _re.split(r"[\s,]+", args.strip()) if x]
        if name == "matrix" and len(v) == 6:
            N = np.array([[v[0], v[2], v[4]], [v[1], v[3], v[5]]])
        elif name == "translate":
            N = np.array([[1., 0., v[0]], [0., 1., v[1] if len(v) > 1 else 0.]])
        elif name == "scale":
            sx = v[0]; sy = v[1] if len(v) > 1 else v[0]
            N = np.array([[sx, 0., 0.], [0., sy, 0.]])
        elif name == "rotate" and len(v) >= 1:
            a = math.radians(v[0]); c, s = math.cos(a), math.sin(a)
            N = np.array([[c, -s, 0.], [s, c, 0.]])
            if len(v) == 3:
                T1 = np.array([[1., 0., v[1]], [0., 1., v[2]]])
                T2 = np.array([[1., 0., -v[1]], [0., 1., -v[2]]])
                N = _mm(_mm(T1, N), T2)
        else:
            continue
        M = _mm(M, N)
    return M

# 4.4.3  apply the composed ancestor transform to a point array
def _apply_tf(M, P):
    P = np.asarray(P, float)
    return np.column_stack([M[0, 0]*P[:, 0] + M[0, 1]*P[:, 1] + M[0, 2],
                            M[1, 0]*P[:, 0] + M[1, 1]*P[:, 1] + M[1, 2]])

# 4.4.4  flatten Bezier segments at flatten_px
def _flatten_cubic(p0, p1, p2, p3, tol):
    """Adaptive cubic flattening; splits until the control polygon is within tol."""
    d1 = np.abs(np.cross(p3 - p0, p0 - p1)); d2 = np.abs(np.cross(p3 - p0, p0 - p2))
    L = np.hypot(*(p3 - p0)) + 1e-12
    n = max(2, int(math.ceil(math.sqrt(max(d1, d2) / L / max(tol, 1e-6) * 3.0))) + 1)
    n = min(n, 64)
    t = np.linspace(0, 1, n)[1:, None]
    return ((1-t)**3)*p0 + 3*((1-t)**2)*t*p1 + 3*(1-t)*(t**2)*p2 + (t**3)*p3

def _parse_path_d(d, tol):
    """SVG 'd' -> [(pts, closed), ...]. M/L/H/V/C/S/Q/T/Z, absolute and relative.
    Beziers flattened at tol (== FUSION_PARAMS.flatten_px, which must be <= fit_tol_px)."""
    toks = _re.findall(r"([MmLlHhVvCcSsQqTtZz])|(-?\d*\.?\d+(?:[eE][-+]?\d+)?)", d or "")
    items = [(a or b) for a, b in toks]
    subs, pts, cur, start, cmd, prev_c2, prev_q = [], [], np.zeros(2), np.zeros(2), None, None, None
    i = 0
    def _num():
        nonlocal i
        v = float(items[i]); i += 1; return v
    while i < len(items):
        if items[i] in "MmLlHhVvCcSsQqTtZz":
            cmd = items[i]; i += 1
        if cmd is None:
            i += 1; continue
        rel = cmd.islower(); C = cmd.upper()
        if C == "M":
            if pts and len(pts) >= 2:
                subs.append((np.array(pts), False))
            x, y = _num(), _num()
            cur = (cur + [x, y]) if rel else np.array([x, y])
            start = cur.copy(); pts = [cur.copy()]
            cmd = "l" if rel else "L"
        elif C == "L":
            x, y = _num(), _num()
            cur = (cur + [x, y]) if rel else np.array([x, y])
            pts.append(cur.copy())
        elif C == "H":
            x = _num(); cur = np.array([cur[0] + x, cur[1]]) if rel else np.array([x, cur[1]])
            pts.append(cur.copy())
        elif C == "V":
            y = _num(); cur = np.array([cur[0], cur[1] + y]) if rel else np.array([cur[0], y])
            pts.append(cur.copy())
        elif C in ("C", "S"):
            if C == "C":
                c1 = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            else:
                c1 = 2*cur - prev_c2 if prev_c2 is not None else cur.copy()
            c2 = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            p3 = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            pts.extend(_flatten_cubic(cur, c1, c2, p3, tol))
            prev_c2 = c2; cur = p3; prev_q = None
        elif C in ("Q", "T"):
            if C == "Q":
                q = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            else:
                q = 2*cur - prev_q if prev_q is not None else cur.copy()
            p2 = (cur + [_num(), _num()]) if rel else np.array([_num(), _num()])
            c1 = cur + 2.0/3.0*(q - cur); c2 = p2 + 2.0/3.0*(q - p2)
            pts.extend(_flatten_cubic(cur, c1, c2, p2, tol))
            prev_q = q; cur = p2; prev_c2 = None
        elif C == "Z":
            if len(pts) >= 2:
                subs.append((np.array(pts), True))
            pts = [start.copy()]; cur = start.copy(); cmd = None
        else:
            i += 1
        if C not in ("C", "S"): prev_c2 = None
        if C not in ("Q", "T"): prev_q = None
    if len(pts) >= 2:
        subs.append((np.array(pts), False))
    return subs

# 4.4.1  recover region codes from id="bnd_A_B_k"
def _meta_from_el(el):
    """(codes, arc_index, side) from data-* if the editor kept them, else from the id."""
    eid = _svg_unescape(el.get("id", ""))
    m = _re.match(r"^bnd_(-?\d+)_(-?\d+)_(\d+)(?:_s(\d+))?$", eid)
    code = arc = side = None
    if m:
        code = (int(m.group(1)), int(m.group(2)))
        arc  = int(m.group(3))
        side = int(m.group(4)) if m.group(4) is not None else None
    a, b = el.get("data-matA"), el.get("data-matB")
    if a is not None and b is not None:
        try: code = (int(a), int(b))
        except ValueError: pass
    if el.get("data-arc") is not None:
        try: arc = int(el.get("data-arc"))
        except ValueError: pass
    return code, arc, side

def _project_onto(P, R):
    """Closest point on polyline R for each point of P (vectorised point-to-segment)."""
    P = np.asarray(P, float); R = np.asarray(R, float)
    A, B = R[:-1], R[1:]
    AB = B - A
    L2 = (AB ** 2).sum(1) + 1e-12
    t = np.clip(((P[:, None, :] - A[None]) * AB[None]).sum(2) / L2[None], 0.0, 1.0)
    proj = A[None] + t[..., None] * AB[None]
    d = np.linalg.norm(P[:, None, :] - proj, axis=2)
    return proj[np.arange(len(P)), d.argmin(axis=1)]

# 4.4.6  collapse a double-line pair back to one line
def _collapse_double_lines(items):
    """Average a double-line pair back onto the true border, keyed by (code, arc, closed)."""
    from collections import defaultdict
    by = defaultdict(list)
    for (code, arc, side, P, closed) in items:
        by[(code, arc, closed)].append((P, side))
    out, n_pairs = [], 0
    for (code, arc, closed), group in by.items():
        sides = {s for _, s in group if s is not None}
        if len(group) == 2 and len(sides) == 2:
            A, B = group[0][0], group[1][0]
            if len(A) == len(B):
                if np.sum((A - B) ** 2) > np.sum((A - B[::-1]) ** 2):
                    B = B[::-1]
                M = 0.5 * (A + B)
            else:
                M = 0.5 * (A + _project_onto(A, B))
            out.append((code, M, closed)); n_pairs += 1
        else:
            for P, _ in group:
                out.append((code, P, closed))
    return out, n_pairs

# 4.4  import corrected lines
def read_qa_svg(svg_path, H_lr, W_si, verbose=True, line_mode=None, double_offset=None,
                params=None, registry_path=None):
    """Parse a QA-edited SVG (Inkscape or Illustrator) back into a BoundaryGraph carrying both region codes on every element."""
    params = params or FUSION_PARAMS
    meta = load_state("trace_meta", default={})
    line_mode     = line_mode     or meta.get("line_mode", "single")
    double_offset = double_offset if double_offset is not None else meta.get("double_offset", 0.35)

    tree = ET.parse(svg_path); root = tree.getroot()
    parent = {c: p for p in root.iter() for c in p}

    def _lname(t): return t.rsplit("}", 1)[-1]
    def _tf_chain(el):
        M, node = np.array([[1., 0., 0.], [0., 1., 0.]]), el
        chain = []
        while node is not None:
            chain.append(node.get("transform")); node = parent.get(node)
        for t in reversed(chain):
            if t: M = _mm(M, _parse_transform(t))
        return M

    items, n_new, n_nocode, n_path, n_poly, n_pairs = [], 0, 0, 0, 0, 0
    for el in root.iter():
        tag = _lname(el.tag)
        if tag not in ("polyline", "path", "polygon"):
            continue
        if (el.get("id") or "").startswith("face_"):     # atlas fills, not boundaries
            continue
        M = _tf_chain(el)
        if tag in ("polyline", "polygon"):
            raw = (el.get("points") or "").strip()
            if not raw: continue
            v = [float(t) for t in _re.split(r"[\s,]+", raw) if t]
            if len(v) < 4: continue
            subs = [(np.array(v, float).reshape(-1, 2), tag == "polygon")]
            n_poly += 1
        else:
            subs = _parse_path_d(el.get("d", ""), params.flatten_px)
            n_path += 1
        if not subs:
            continue
        code, arc_i, side = _meta_from_el(el)
        for si, (P, closed) in enumerate(subs):
            if len(P) < 2: continue
            vox = _unorient_arr(_apply_tf(M, P), H_lr, W_si)      # display -> voxel
            if code is None:
                n_nocode += 1
                if closed or (len(vox) > 3 and np.hypot(*(vox[0] - vox[-1])) < 1.0):
                    items.append((None, None, None, vox, True))
                continue
            items.append((tuple(sorted(code)), (arc_i, si), side, vox, bool(closed)))

    coded   = [it for it in items if it[0] is not None]
    uncoded = [(None, P, True) for (c, a, s, P, cl) in items if c is None]
    if line_mode == "double":
        collapsed, n_pairs = _collapse_double_lines(coded)
    else:
        collapsed = [(c, P, cl) for (c, a, s, P, cl) in coded]
    collapsed += uncoded

    reg = ef.load_registry(registry_path or REGISTRY_PATH)
    # 4.4.5  a closed path with no code becomes a new expert region id; an open path with no code is ignored but counted
    next_new = ef.registry_next_new(reg, params)
    final = []
    for (code, P, closed) in collapsed:
        if code is None:
            code = (params.outer_code, next_new); next_new += 1; n_new += 1
        final.append((tuple(sorted(code)), P, closed))

    g = bg.BoundaryGraph(nodes=np.empty((0, 2)), elements=[])
    index, xy = {}, []
    def _nid(p):
        k = (round(float(p[0]), 6), round(float(p[1]), 6))
        if k not in index:
            index[k] = len(xy); xy.append((float(p[0]), float(p[1])))
        return index[k]
    for (a, b), P, closed in final:
        Q = np.vstack([P, P[0]]) if (closed and np.hypot(*(P[0] - P[-1])) > 1e-9) else P
        prev = None
        for p in Q:
            ni = _nid(p)
            if prev is not None and prev != ni:
                g.elements.append([prev, ni, int(a), int(b)])
            prev = ni
    g.nodes = np.asarray(xy, float) if xy else np.zeros((0, 2))

    if verbose:
        msg = (f"    {os.path.basename(svg_path)}: {n_poly} polyline(s) + {n_path} path(s) -> "
               f"{len(g.elements)} elements, {len(ef.graph_to_arcs(g))} arcs")
        if n_pairs:  msg += f", {n_pairs} double-line pair(s) collapsed"
        if n_new:    msg += f", {n_new} NEW expert region(s) from id {params.new_id_start}"
        if n_nocode and not n_new: msg += f", {n_nocode} open shape(s) with no code IGNORED"
        print(msg)
    return g

print("SVG round-trip machinery ready: graph_to_svg (writer), read_qa_svg (reader).")


In [ ]:
# =====================================================================================
# 3.2  EXPORT SLICES FOR CORRECTION
#
# 3.2.1  warped T1 underlay; 3.2.2 re-mask to the chosen hemisphere
# 3.2.3  write the layered SVG; 3.2.4 write the region reference table
# 3.2.6  round-trip gate on the representative slice (codes survive, geometry in tol)
# 3.2.7  each export is copied into RUN_DIR/qc as the fusion input
#
# Nothing downstream requires this stage: fusion reads the boundary graphs directly.
# EXPORT_SVGS and EXPORT_EVERY_N (0.4.5) control how much is written.
# =====================================================================================
import os, shutil, csv
import numpy as np, nibabel as nib
from tqdm.auto import tqdm

# 3.2.5  output the QA SVGs
def export_qa_slices(every_n=None, subjects=None, stroke=EXPORT_STROKE):
    """Write one SVG per subject per exported slice, plus a region reference table."""
    if not EXPORT_SVGS:
        print("EXPORT_SVGS=False -> skipped. Fusion reads the graphs directly.")
        return []
    every_n = every_n or EXPORT_EVERY_N
    base = f"{RUN_DIR}/lines_export_for_qa"
    slices_dir, tables_dir = f"{base}/slices", f"{base}/tables"
    os.makedirs(slices_dir, exist_ok=True); os.makedirs(tables_dir, exist_ok=True)

    prop = load_state("propagated"); regs = load_state("regs"); meta = load_state("trace_meta")
    table   = load_region_table(TEMPLATE_LUT)
    lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    sids = subjects or list(prop)
    todo = [s for i, s in enumerate(meta["slices"]) if i % every_n == 0]
    first_sid = sids[0]
    written = []

    for sid in sids:
        # 3.2.1  warped T1 underlay (resampled onto the working grid, not deformed)
        t1_vol = nib.load(regs[sid]["warped"]).get_fdata() if sid in regs else None
        for sidx in tqdm(todo, desc=f"export {sid}", unit="slice"):
            g = bg.read_ascii(prop[sid][sidx])
            # 3.2.2  re-mask the label slice to the chosen hemisphere
            lab_slice, _ = _hemisphere_mask(np.take(lab_vol, sidx, axis=SLICE_AXIS),
                                            meta["hemisphere"])
            # 3.2.3.3  label anchor points
            anchors = _label_points_on_slice(lab_slice)
            svg = f"{slices_dir}/{sid}_slice_{sidx:03d}.svg"
            graph_to_svg(g, svg, t1_volume=t1_vol, slice_index=sidx, axis=SLICE_AXIS,
                         label_anchors=anchors, table=table, stroke=stroke,
                         line_mode=meta["line_mode"], double_offset=meta["double_offset"],
                         label_detail=meta["label_detail"], hemisphere=meta["hemisphere"],
                         midline_lr=meta["midline_lr"])
            written.append(svg)
            # 3.2.7  auto-copy each export into RUN_DIR/qc
            if AUTO_QA_COPY:
                shutil.copyfile(svg, f"{RUN_DIR}/qc/{sid}_slice_{sidx:03d}.svg")
            # 3.2.4  the region table is written regardless of the qc copy toggle
            if sid == first_sid and meta["label_detail"] != "none":
                region_ids, _ = _number_regions(anchors)
                with open(f"{tables_dir}/slice_{sidx:03d}_regions.csv", "w", newline="") as f:
                    # 3.2.4  region reference table
                    w = csv.writer(f)
                    w.writerow(["abbreviation", "name", "id", "color_hex", "type"])
                    for rid in region_ids:
                        # one ROW per fragment (its own code), one NAME and COLOUR per base
                        base_id = ef.fragment_base(rid); fx = ef.fragment_index(rid)
                        info = table.get(base_id, {})
                        ab = info.get("abbrev", str(base_id)); nm = info.get("name", ab)
                        if fx:
                            ab = f"{ab}#{fx}"; nm = f"{nm} (fragment {fx})"
                        col = _rgb_hex(info.get("rgb", region_color(rid)))
                        kind = combined_lut.get(base_id, {}).get("kind", "region")
                        w.writerow([ab, nm, rid, col, kind])

    # round-trip gate on the representative slice: do the codes and the geometry survive?
    rep = meta["slice_index"] if meta["slice_index"] in todo else todo[0]
    g0  = bg.read_ascii(prop[first_sid][rep])
    H_lr, W_si = np.take(lab_vol, rep, axis=SLICE_AXIS).shape
    # 3.2.6 / 3.2.6.1  re-import gate: every region code must survive and the geometry stay in tolerance
    g_rt = read_qa_svg(f"{slices_dir}/{first_sid}_slice_{rep:03d}.svg", H_lr, W_si, verbose=False)
    codes_out = {tuple(sorted((int(e[2]), int(e[3])))) for e in g0.elements}
    codes_in  = {tuple(sorted((int(e[2]), int(e[3])))) for e in g_rt.elements}
    d_rt = ef._curve_dist(ef.graph_to_arcs(g_rt), ef.graph_to_arcs(g0), 0.5)
    checks = [
        ("every region code survives the SVG round-trip", codes_out <= codes_in,
         f"missing on re-import: {sorted(codes_out - codes_in)[:6]}"),
        (f"round-trip geometry error {d_rt:.4f}px within tolerance",
         d_rt <= max(2 * (FUSION_PARAMS.grid_write or 0.01), 0.05),
         "EXPORT_SMOOTH_SIGMA > 0, or the display transform is not inverting"),
    ]
    report_outputs("CELL 8 export QA SVGs", files=written[:4], checks=checks)
    print(f"\nExported {len(written)} SVG(s) (every {every_n}th slice) -> {slices_dir}")
    print(f"  line_mode='{meta['line_mode']}'  label_detail='{meta['label_detail']}'  "
          f"hemisphere='{meta['hemisphere']}'  export_smooth_sigma={EXPORT_SMOOTH_SIGMA}")
    print(f"  codes carried in id='bnd_A_B_k' (Illustrator strips data-*, keeps id).")
    return written

def preview_qa_slice(sid=None, sidx=None, stroke=0.4, scale=2.0):
    """DIAGNOSTIC: render one exported slice inline."""
    from IPython.display import HTML, display
    import tempfile
    prop = load_state("propagated"); regs = load_state("regs"); meta = load_state("trace_meta")
    sid  = sid or next(iter(prop))
    sidx = meta["slice_index"] if sidx is None else sidx
    table   = load_region_table(TEMPLATE_LUT)
    lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    g = bg.read_ascii(prop[sid][sidx])
    t1_vol = nib.load(regs[sid]["warped"]).get_fdata() if sid in regs else None
    lab_slice, _ = _hemisphere_mask(np.take(lab_vol, sidx, axis=SLICE_AXIS),
                                    meta.get("hemisphere", "whole"))
    anchors = _label_points_on_slice(lab_slice)
    tmp = tempfile.NamedTemporaryFile(suffix=".svg", delete=False).name
    graph_to_svg(g, tmp, t1_volume=t1_vol, slice_index=sidx, axis=SLICE_AXIS,
                 label_anchors=anchors, table=table, stroke=stroke,
                 line_mode=meta["line_mode"], double_offset=meta["double_offset"],
                 label_detail=meta["label_detail"],
                 hemisphere=meta.get("hemisphere", "whole"), midline_lr=meta.get("midline_lr"))
    svg_text = open(tmp).read()
    if svg_text.startswith("<?xml"):
        svg_text = svg_text.split("?>", 1)[1]
    print(f"preview subject={sid} slice={sidx}")
    display(HTML(f'<div style="background:#888;display:inline-block;padding:6px;'
                 f'transform:scale({scale});transform-origin:top left">{svg_text}</div>'))
    return tmp

export_qa_slices()
preview_qa_slice()


## Stage 4 - Expert correction and alignment (manual)

Open the SVGs from `RUN_DIR/lines_export_for_qa/slices/` in Illustrator or Inkscape.

- The **background** layer (white page + T1 raster) is locked. Only the **boundaries**
  layer is editable.
- Adjust the lines to fit each scan's own anatomy. Add or delete regions as needed.
- **Template graphs must remain topologically identical across subjects.** Where they
  diverge, fusion refuses the slice rather than averaging across a topology change.
- Region codes travel in the element `id` (`bnd_A_B_k`). Illustrator strips unknown
  `data-*` attributes on save but keeps `id`, so do not rename elements.
- A closed path with no parseable code is read as a region you drew and gets a fresh id
  from 9001 up. An open path with no code cannot bound anything and is ignored (and
  counted, so it is never silently swallowed).

Save the edited files into `RUN_DIR/qc/` as `<SID>_slice_<NNN>.svg`, then set
`USE_SVG_INPUTS = True` in CELL 9.

## Stage 5 - Contour fusion

Node matching across subjects, then per-arc averaging between matched anchors, then
polygon construction. The polygons are the atlas.

In [ ]:
# =====================================================================================
# 5  CONTOUR FUSION, ALL SLICES
#
# 5.1  setup: fusion params, per-subject boundary graphs, seeds, expected region ids
# 5.2  construct the per-subject line network (arcs + nodes, first pass)
# 5.3  inject phantom regions (point / line / tree reductions)
# 5.4  decompose again with the phantoms in place
# 5.5  node fusion; 5.6 per-arc fusion (edge fusion)
# 5.7  polygon construction, face assignment, per-slice registry
# 5.8  diagnostics S1-S7
# =====================================================================================
import os, glob, time
import numpy as np, nibabel as nib
from tqdm.auto import tqdm

FUSION_INPUT_PATHS = {}                 # optional per-subject overrides {sid: path}
# actually been edited in Illustrator -- the round trip exists to capture those edits.

# 5.1.2  locate each subject's fusion input for one slice
def _resolve_fusion_inputs(sids, sidx, input_dir=None, input_paths=None):
    """One SVG path per subject: explicit override, then this run's file, then newest."""
    input_dir   = input_dir   or FUSION_INPUT_DIR
    input_paths = input_paths if input_paths is not None else FUSION_INPUT_PATHS
    resolved = {}
    for sid in sids:
        if sid in input_paths:
            p = input_paths[sid]
        elif os.path.exists(f"{input_dir}/{sid}_slice_{sidx:03d}.svg"):
            p = f"{input_dir}/{sid}_slice_{sidx:03d}.svg"
        else:
            cands = glob.glob(f"{input_dir}/*{sid}_slice_{sidx:03d}.svg")
            if not cands:
                raise FileNotFoundError(
                    f"{sid}: no fusion input SVG in {input_dir} for slice {sidx:03d}. "
                    f"Run CELL 8, or set FUSION_INPUT_PATHS[{sid!r}].")
            p = max(cands, key=os.path.getmtime)
        resolved[sid] = p
    return resolved

# 5.1.2  load the per-subject boundary graphs
def _graphs_for_slice(sidx, use_svg=None):
    """{sid: BoundaryGraph} for one slice, from the QA SVGs or straight from the graphs."""
    use_svg = USE_SVG_INPUTS if use_svg is None else use_svg
    if not use_svg:
        prop = load_state("propagated")
        return {sid: bg.read_ascii(sl[sidx]) for sid, sl in prop.items() if sidx in sl}
    lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    H_lr, W_si = np.take(lab_vol, sidx, axis=SLICE_AXIS).shape
    sids = list(load_state("propagated"))
    resolved = _resolve_fusion_inputs(sids, sidx)
    return {sid: read_qa_svg(p, H_lr, W_si, verbose=False) for sid, p in resolved.items()}

# 5.1 - 5.7  run contour fusion over every traced slice
def fuse_all_slices(use_svg=None, limit=None, verbose=False):
    """Run ef.run_fusion over every traced slice."""
    prop = load_state("propagated"); meta = load_state("trace_meta")
    hemi = meta.get("hemisphere", "whole")
    all_slices = meta["slices"] if limit is None else meta["slices"][:limit]
    lab_vol  = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    # 5.7.4  per-slice region registry
    registry = ef.load_registry(REGISTRY_PATH)

    def _acr_for(code):                       # fragment-aware acronym: "V1" or "V1#1"
        b, fx = ef.fragment_base(code), ef.fragment_index(code)
        ab = (combined_lut.get(int(b), {}) or {}).get("abbrev", str(int(b)))
        return ab if fx == 0 else f"{ab}#{fx}"
    acr_map  = {int(rid): _acr_for(rid) for rid in combined_lut}

    fused_graphs, fused_regions, fused_anchors, all_diag = {}, {}, {}, {}
    n_unl = n_mrg = n_refused = 0
    refused = []
    t0 = time.time()
    for sidx in tqdm(all_slices, desc="fuse", unit="slice"):
        graphs = _graphs_for_slice(sidx, use_svg)
        if not graphs:
            continue
        # 5.1.3.1  mask to the chosen hemisphere
        lab_slice, _ = _hemisphere_mask(np.take(lab_vol, sidx, axis=SLICE_AXIS), hemi)
        # FRAGMENTS [edge_fusion 0b]: relabel AFTER masking, then put every subject's codes
        # onto the template's canonical fragments before anything is matched or fused.
        # 5.1.3.2  fragment relabelling, then canonicalisation onto the template's fragments
        lab_slice = ef.fragment_relabel_slice(lab_slice)
        # 5.1.3.2  match each subject's fragments onto the template's by centroid distance
        for (_s, _o, _n) in ef.canonicalize_fragments(
                graphs, ef.fragment_canon_table(lab_slice), FUSION_PARAMS):
            print(f"  slice {sidx:03d} [fragments] {_s}: {_o} -> {_n}")
        # 5.1.3.3  per-region seeds
        seeds    = _seed_points_on_slice(lab_slice)
        # 5.1.3.4  non-zero region ids present on this slice
        expected = sorted(set(int(r) for r in np.unique(lab_slice)) - {0})
        acr_map.update({int(r): _acr_for(r) for r in expected})
        try:
            # 5.2 - 5.7  line network, phantom injection, node fusion, arc fusion, polygon construction
            out = ef.run_fusion(graphs, seeds, expected, weights=FUSION_WEIGHTS,
                                params=FUSION_PARAMS, registry=registry,
                                run_id=FUSION_RUN_ID, slice_index=sidx, verbose=verbose,
                                report_ctx={"run_label": RUN_NAME, "slice_index": sidx,
                                            "acronyms": acr_map})
        # 5.3.3.4  complex case: the slice is refused rather than emitted
        except ef.TopologyDivergence as e:
            # The subjects' region-adjacency graphs disagree. Averaging across a topology
            # change produces a locally wrong map with no warning, so this slice is refused
            # rather than emitted.
            n_refused += 1; refused.append(sidx)
            print(f"  slice {sidx:03d}: FUSION REFUSED (subject topology disagrees)")
            continue
        registry            = out["registry"]
        fused_graphs[sidx]  = (out["graph"], seeds)
        fused_regions[sidx] = out["regions"]
        fused_anchors[sidx] = out["anchors"]
        all_diag[sidx]      = out["diag"]
        n_unl += len([f for f in out["flags"] if f[0] == "unlabeled"])
        n_mrg += len([f for f in out["flags"] if f[0] == "merged_regions"])

    # 5.7.4  write the registry
    ef.save_registry(registry, REGISTRY_PATH)
    save_state("fused_graphs",  fused_graphs)
    save_state("fused_regions", fused_regions)
    save_state("fused_anchors", fused_anchors)
    save_state("fusion_diag",   all_diag)

    n_faces = sum(sum(len(v) for v in r.values()) for r in fused_regions.values())
    n_arcs  = sum(len(ef.graph_to_arcs(g)) for g, _ in fused_graphs.values())
    euler_bad = [s for s, d in all_diag.items() if d.get("G_euler_defect", 0) != 0]
    fallback  = [s for s, d in all_diag.items() if d.get("S5_fallback_chainer", False)]
    checks = [
        ("every slice fused", len(fused_graphs) == len(all_slices),
         f"{n_refused} slice(s) refused: {refused[:12]}"),
        ("no UNLABELED faces (no QA gaps)", n_unl == 0,
         "magenta faces become a stray material in the volume -- redraw the missing boundary"),
        ("no merged regions", n_mrg == 0, "a separating boundary is missing"),
        ("polygonize did not fall back", not fallback, f"slices: {fallback[:12]}"),
        ("Euler identity holds (V-E+F == C)", not euler_bad,
         f"an arc, node or face was lost or duplicated on slices {euler_bad[:12]}"),
    ]
    report_outputs("CELL 9 fuse (all slices)", files=[REGISTRY_PATH],
                   state_keys=["fused_graphs", "fused_regions", "fused_anchors", "fusion_diag"],
                   checks=checks)
    print(f"\nFused {len(fused_graphs)}/{len(all_slices)} slices in {time.time()-t0:.1f}s")
    print(f"  {n_arcs} fused arcs -> {n_faces} faces, {n_unl} unlabeled, {n_refused} refused")
    if not FUSION_PARAMS.run_comparative_diag:
        print("  run_comparative_diag=False: S6/S7 shape metrics skipped (polygons identical).")
    return fused_regions

# 5.8.1  S1-S5 correctness table for one slice
def print_fusion_report(sidx=None, verbose=False):
    """DIAGNOSTIC: the S1..S7 table for one slice."""
    diag = load_state("fusion_diag")
    sidx = load_state("trace_meta")["slice_index"] if sidx is None else sidx
    if sidx not in diag:
        print(f"no diagnostics for slice {sidx}"); return []
    return ef.print_report(diag[sidx], f"slice {sidx}", verbose=verbose)

# 5.8.2  S6/S7 comparative shape metrics
def fusion_summary():
    """DIAGNOSTIC: the diagnostics that vary slice to slice, aggregated over the run."""
    diag = load_state("fusion_diag")
    if not diag:
        print("no fusion diagnostics"); return
    keys = ["S3_node_match_rate", "S3_node_disp_p95_px", "S3_node_rejected_far",
            "S4_arcs_dropped", "S4_snap_dist_max_px", "S4_pts_total",
            "S7_budget_saturated", "G_euler_defect"]
    print(f"{'metric':26s} {'min':>10s} {'median':>10s} {'max':>10s}")
    for k in keys:
        vals = [d[k] for d in diag.values() if isinstance(d.get(k), (int, float))]
        if not vals: continue
        print(f"{k:26s} {min(vals):10.4g} {float(np.median(vals)):10.4g} {max(vals):10.4g}")
    sug = [d["S3_suggested_node_match_max"] for d in diag.values()
           if isinstance(d.get("S3_suggested_node_match_max"), (int, float))]
    if sug:
        print(f"\n  suggested node_match_max (3 x measured p95): {max(sug):.2f}"
              f"   you have {FUSION_PARAMS.node_match_max}")

fuse_all_slices()
print_fusion_report()
fusion_summary()


## Stage 6 - Conversion to volumetric model

Mirror back to a bilateral atlas, rasterise into a labelled volume, marching-cubes it into
per-region surfaces, condition those surfaces, and emit the deliverables.

In [ ]:
# =====================================================================================
# 6.1  MIRROR THE HEMISPHERE BACK FOR BILATERAL SYMMETRY
#
# 6.1.1  reflect the kept-hemisphere graph across the L-R midline
# 6.1.2  weld the reflection to the original: merge seam nodes, drop duplicate arcs
# 6.1.3  rebuild full-brain faces from the welded graph
# 6.1.5  fragments are merged back to their base region ids first
# =====================================================================================
import numpy as np, nibabel as nib
from tqdm.auto import tqdm

# 6.1.1  reflect the kept-hemisphere graph across the L-R midline
def mirror_hemisphere_graph(g, midline_lr, weld_tol=MIRROR_WELD_TOL):
    """Reflect a single-hemisphere graph across the L-R midline and weld it to the original.
    Node coords are (x=col=S-I, y=row=L-R); the L-R coordinate is mirrored."""
    arcs = ef.graph_to_arcs(g)
    out = []
    for a in arcs:
        out.append(a)
        P = np.asarray(a.pts, float).copy()
        # 6.1.1  mirror the L-R coordinate
        P[:, 1] = 2.0 * midline_lr - P[:, 1]
        near = np.abs(P[:, 1] - midline_lr) <= weld_tol
        P[near, 1] = midline_lr
        out.append(ef.Arc(a.code, P, a.closed, sid="mirror"))
    # 6.1.2 / 6.1.2.1  weld: snap seam nodes onto the midline so they merge
    for a in out:
        P = np.asarray(a.pts, float)
        near = np.abs(P[:, 1] - midline_lr) <= weld_tol
        P[near, 1] = midline_lr
        a.pts = P
    # 6.1.2.2  duplicate midline arcs are dropped when the arcs are rebuilt into a graph
    return ef.arcs_to_graph(out)

# 6.1  mirror the hemisphere back for bilateral symmetry
def mirror_back_all_slices(source="fused", rebuild_polygons=None):
    """source='fused' -> CELL 9's fused graphs. HEMISPHERE='whole' passes through."""
    if rebuild_polygons is None:
        rebuild_polygons = FUSION_PARAMS.build_polygons
    meta = load_state("trace_meta")
    hemi = meta.get("hemisphere", "whole")
    rep  = meta["slice_index"]

    if source == "fused":
        graphs = {s: gv[0] for s, gv in load_state("fused_graphs").items()}
    else:
        raise ValueError("source must be 'fused'")

    lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    atlas_full, atlas_regions = {}, {}

    if hemi == "whole":
        print("HEMISPHERE='whole' -> nothing to mirror; passing graphs through unchanged.")
        fused_regions = load_state("fused_regions", default={})
        for s, g in graphs.items():
            outp = f"{RUN_DIR}/atlas/atlas_full_slice_{s:03d}.txt"
            bg.write_ascii(g, outp); atlas_full[s] = outp
            # fragments are ONE region in the finished atlas (base ids)
            # 6.1.5  fragments merged back to their base region ids
            atlas_regions[s] = ef.merge_fragment_regions(fused_regions.get(s, {}))
        save_state("atlas_lines_full", atlas_full)
        save_state("atlas_regions_full", atlas_regions)
        report_outputs("CELL 10 mirror-back (pass-through)",
                       state_keys=["atlas_lines_full", "atlas_regions_full"])
        return atlas_full

    midline = meta["midline_lr"]
    n_unl = 0
    for s, g in tqdm(sorted(graphs.items()), desc="mirror", unit="slice"):
        full = mirror_hemisphere_graph(g, midline)
        outp = f"{RUN_DIR}/atlas/atlas_full_slice_{s:03d}.txt"
        bg.write_ascii(full, outp)
        atlas_full[s] = outp
        if rebuild_polygons:
            # Mirroring is a topology change (the halves now share the midline), so the faces
            # are re-derived from the welded graph rather than mirrored. Seeds come from the
            # SAME relabelled half-slice the fusion used, then mirrored: the bilateral slice
            # has a DIFFERENT fragment decomposition from the one the arcs carry.
            sl, _mid = _hemisphere_mask(np.take(lab_vol, s, axis=SLICE_AXIS), hemi)
            sl = ef.fragment_relabel_slice(sl)
            seeds = _seed_points_on_slice(sl)
            seeds = seeds + [((x, 2.0 * midline - y), rid) for ((x, y), rid) in seeds]
            # 6.1.3  rebuild full-brain faces from the welded graph
            rb = ef.rebuild_atlas(ef.graph_to_arcs(full), seeds, FUSION_PARAMS)
            # 6.1.4 / 6.1.5  one base region id shared by both hemispheres
            atlas_regions[s] = ef.merge_fragment_regions(rb["regions"])
            n_unl += len([f for f in rb["flags"] if f[0] == "unlabeled"])

    if not rebuild_polygons:
        print("  [build_polygons=False] mirrored LINES only; atlas_regions_full is empty.")
    save_state("atlas_lines_full", atlas_full)
    save_state("atlas_regions_full", atlas_regions)

    # weld gate on the representative slice
    g_half = graphs[rep]; g_full = bg.read_ascii(atlas_full[rep])
    yv = np.asarray(g_full.nodes, float)[:, 1]
    # 6.1.2.1  weld gate: count the shared midline vertices
    n_seam = int((np.abs(yv - midline) < 1e-6).sum())
    merged = 2 * len(g_half.nodes) - len(g_full.nodes)
    # Counting graph COMPONENTS would be wrong: an island is a closed loop and therefore its
    # own component by design. What matters is that the halves SHARE their midline vertices,
    welded = len(g_full.nodes) < 2 * len(g_half.nodes)
    n_faces = sum(sum(len(v) for v in r.values()) for r in atlas_regions.values())
    checks = [
        (f"full slice ~2x half nodes ({len(g_half.nodes)} -> {len(g_full.nodes)})",
         len(g_full.nodes) >= 1.6 * len(g_half.nodes) - 2, "mirror did not duplicate nodes"),
        ("nodes present on BOTH sides of the midline",
         (yv < midline).any() and (yv > midline).any(),
         "reflection axis wrong -> check midline_lr / HEMISPHERE"),
        (f"the halves are WELDED at the midline ({n_seam} seam nodes, {merged} merged)",
         welded and n_seam > 0,
         "raise MIRROR_WELD_TOL: seam nodes are not landing on identical coordinates, so "
         "polygonize will not close faces across the midline"),
        ("no UNLABELED faces after the rebuild", n_unl == 0,
         "magenta faces = QA gaps; they become a stray material in the volume"),
    ]
    report_outputs("CELL 10 mirror-back", state_keys=["atlas_lines_full", "atlas_regions_full"],
                   checks=checks)
    print(f"\nMirrored {len(atlas_full)} slices (hemisphere='{hemi}', midline L-R={midline:.1f})")
    if rebuild_polygons:
        print(f"  rebuilt {n_faces} full-brain faces across "
              f"{len(atlas_regions)} slices, {n_unl} unlabeled")
    return atlas_full

mirror_back_all_slices(source="fused")


In [ ]:
# =====================================================================================
# 6.2.7  FILLED-ATLAS PREVIEW AND EXPORT
#
# 6.2.7.1  rendered over a subject underlay (UNDERLAY_SUBJECT, 0.4.7)
# 6.2.7.2  coloured variant (LUT colours) and normal variant (region ids)
#
# Falls back to the single-hemisphere fusion when 6.1 has not been run.
# =====================================================================================
import os, time, tempfile
import numpy as np, nibabel as nib
from IPython.display import HTML, display
from tqdm.auto import tqdm

_ATLAS = {}

# 6.2.7.1  subject underlay for the preview
def _underlay(sid=None):
    sid  = UNDERLAY_SUBJECT if sid is None else sid
    regs = load_state("regs", default={}) or {}
    if sid and sid in regs:
        return nib.load(regs[sid]["warped"]).get_fdata(), sid
    return nib.load(TEMPLATE_T1).get_fdata(), "DB09 template"

# 6.2.7  load the mirrored atlas lines and regions
def _atlas_cache(reload=False):
    """Load the mirrored full-brain atlas once; fall back to the single-hemisphere fusion."""
    if _ATLAS and not reload:
        if _ATLAS.get("under_sid") != UNDERLAY_SUBJECT:
            _ATLAS["t1"], _ATLAS["under"] = _underlay()
            _ATLAS["under_sid"] = UNDERLAY_SUBJECT
            print(f"underlay -> {_ATLAS['under']}")
        return _ATLAS
    lines = load_state("atlas_lines_full",   default={}) or {}
    regs  = load_state("atlas_regions_full", default={}) or {}
    if any(regs.values()):
        which = "atlas_*_full (mirrored full brain)"
    else:
        fg    = load_state("fused_graphs",  default={}) or {}
        lines = {s: gv[0] for s, gv in fg.items()}
        regs  = load_state("fused_regions", default={}) or {}
        which = "fused_* (SINGLE hemisphere -- run CELL 10 to mirror)"
    # the atlas VIEW is per base region: fragments merge back into one region here.
    # merge_fragment_regions is idempotent, so this is safe on either source above.
    regs  = {s: ef.merge_fragment_regions(r) for s, r in regs.items()}
    if not any(regs.values()):
        raise RuntimeError("No filled atlas in state. Set FUSION_PARAMS.build_polygons=True "
                           "in CELL 0, then re-run CELL 9 (fuse) and CELL 10 (mirror).")
    t1, under = _underlay()
    _ATLAS.update(lines=lines, regions=regs, which=which, t1=t1, under=under,
                  under_sid=UNDERLAY_SUBJECT,
                  lab=nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32),
                  table=load_region_table(TEMPLATE_LUT),
                  meta=load_state("trace_meta"))
    print(f"atlas cache: {len(regs)} slices  [{which}]   underlay = {under}")
    return _ATLAS

def _atlas_graph(sidx):
    v = _atlas_cache()["lines"].get(sidx)
    if v is None: return None
    return bg.read_ascii(v) if isinstance(v, str) else v

def _render_atlas(sidx, filled, fill_op, stroke, out_path):
    a = _atlas_cache(); g = _atlas_graph(sidx); regions = a["regions"].get(sidx)
    if g is None or not regions:
        return None, None
    anchors = _label_points_on_slice(np.take(a["lab"], sidx, axis=SLICE_AXIS))
    graph_to_svg(g, out_path, t1_volume=a["t1"], slice_index=sidx, axis=SLICE_AXIS,
                 label_anchors=anchors, table=a["table"], stroke=stroke,
                 line_mode="single", double_offset=a["meta"].get("double_offset", 0.35),
                 label_detail=a["meta"].get("label_detail", "abbrev"),
                 hemisphere="whole", midline_lr=None, smooth_sigma=0.0,
                 regions=(regions if filled else None), fill_opacity=fill_op)
    svg = open(out_path).read()
    return (svg.split("?>", 1)[1] if svg.startswith("<?xml") else svg), regions

# 6.2.7  inline preview
def preview_atlas(sidx=None, step=None, fill=None, scale=None, stroke=0.36,
                  variants=EXPORT_VARIANTS):
    """preview_atlas() | preview_atlas(200) | preview_atlas(step=25)"""
    a = _atlas_cache()
    fill  = ATLAS_PREVIEW_FILL  if fill  is None else fill
    scale = ATLAS_PREVIEW_SCALE if scale is None else scale
    slices  = sorted(a["regions"])
    targets = (slices[::step] if step
               else [a["meta"]["slice_index"] if sidx is None else sidx])
    tmp = tempfile.gettempdir()
    for s in targets:
        panels, regions = [], None
        for v in variants:
            svg, regions = _render_atlas(s, v == "colored",
                                         fill if v == "colored" else 0.0, stroke,
                                         f"{tmp}/_pv_{v}_{s:03d}.svg")
            if svg is None: break
            panels.append((("COLORED (filled faces = the polygonize output)" if v == "colored"
                            else "NORMAL (boundary lines only)"), svg))
        if not panels:
            print(f"slice {s}: no atlas"); continue
        unl = sorted(r for r in regions
                     if FUSION_PARAMS.unlabeled_id_start <= r < FUSION_PARAMS.new_id_start)
        n_faces = sum(len(v) for v in regions.values())
        print(f"slice {s}: {len(regions)} regions, {n_faces} faces   on {a['under']}"
              + (f"   !! UNLABELED {unl}" if unl else ""))
        for title, txt in panels:
            display(HTML(f'<div style="font:bold 13px sans-serif;margin:12px 0 2px">'
                         f'slice {s} &mdash; {title} &mdash; on {a["under"]}</div>'
                         f'<div style="background:#888;display:inline-block;padding:6px;'
                         f'transform:scale({scale});transform-origin:top left">{txt}</div>'))

# 6.2.7 / 6.2.7.2  coloured and normal variants
def export_filled_atlas(every_n=None, variants=None, fill=None, stroke=0.36, limit=None):
    """Write the colored/normal atlas SVGs. Stage 6 builds the volume from state and does
    NOT need these files."""
    if not EXPORT_FILLED_SVGS:
        print("EXPORT_FILLED_SVGS=False -> preview only.")
        return []
    a = _atlas_cache()
    every_n  = every_n  or EXPORT_ATLAS_EVERY_N
    variants = variants or EXPORT_VARIANTS
    fill     = ATLAS_PREVIEW_FILL if fill is None else fill
    base = f"{RUN_DIR}/lines_export_for_qa"
    dirs = {v: f"{base}/{v}" for v in variants}
    for d in dirs.values():
        os.makedirs(d, exist_ok=True)
    tag  = f"_{a['under']}" if a["under"] != "DB09 template" else ""
    todo = sorted(a["regions"])[::every_n]
    if limit: todo = todo[:limit]
    t0, written = time.time(), []
    for s in tqdm(todo, desc="atlas svg", unit="slice"):
        for v in variants:
            stem = "atlas_filled" if v == "colored" else "atlas_lines"
            dst  = f"{dirs[v]}/{stem}{tag}_slice_{s:03d}.svg"
            svg, _ = _render_atlas(s, v == "colored", fill if v == "colored" else 0.0,
                                   stroke, dst)
            if svg is not None: written.append(dst)
    print(f"\nWrote {len(written)} file(s) in {time.time()-t0:.1f}s -> {base}")
    return written

preview_atlas()
export_filled_atlas()


In [ ]:
# =====================================================================================
# 5.8.3 / 5.8.4  VERIFICATION
#
# 5.8.3  S7_sweep_identity: all weight on subject i must return subject i's own map,
#        within fit_tol*3 + node_tol/2. Needs no reference data.
# 5.8.4  M8 weight sweep: the fused output tracked across the weight simplex.
# =====================================================================================
import dataclasses as _dc
import numpy as np, nibabel as nib

meta  = load_state("trace_meta")
sidx  = meta["slice_index"] if VERIFY_SLICE is None else VERIFY_SLICE
graphs = _graphs_for_slice(sidx)
print(f"verifying on {len(graphs)} subject(s), slice {sidx}\n")

# --- 1) S7_sweep_identity -------------------------------------------------------------
si = ef.sweep_identity(graphs, FUSION_PARAMS)
print("=== S7_sweep_identity ===")
print(f"  worst deviation                : {si['S7_sweep_identity_px']:.4f} px")
print(f"  bound (fit_tol*3 + node_tol/2) : {si['S7_sweep_identity_bound_px']:.4f} px")
print(f"  per subject                    : {si['S7_sweep_identity_per_subject']}")
print(f"  {'PASS' if si['S7_sweep_identity_ok'] else 'FAIL'}")

d0 = load_state("fusion_diag").get(sidx, {})
if d0.get("S7_budget_saturated", 0) > 0:
    print(f"  !! S7_budget_saturated = {d0['S7_budget_saturated']}: segments hit "
          f"n_max_seg={FUSION_PARAMS.n_max_seg} while still over fit_tol. Lower kp_alpha "
          f"(currently {FUSION_PARAMS.kp_alpha}) or raise n_max_seg.")

# --- 2) M8 weight-sweep stability curve ----------------------------------------------
# At w=0 and w=1 the inputs must be recovered; non-monotone metrics in between indicate
# instability. Every column is an S6/S7 shape metric, so this needs the polygon rebuild.
if not FUSION_PARAMS.build_polygons:
    print("\n=== M8 weight sweep: SKIPPED (build_polygons=False) ===")
elif len(graphs) >= 2:
    VERIFY_PARAMS = ef.FusionParams(**{**_dc.asdict(FUSION_PARAMS),
                                       "run_comparative_diag": True, "metrics_avg_gl": True})
    lab_vol = nib.load(TEMPLATE_LABELS).get_fdata().astype(np.int32)
    lab_slice, _ = _hemisphere_mask(np.take(lab_vol, sidx, axis=SLICE_AXIS),
                                    meta.get("hemisphere", "whole"))
    seeds    = _seed_points_on_slice(lab_slice)
    expected = sorted(set(int(r) for r in np.unique(lab_slice)) - {0})
    pair = (list(graphs)[0], list(graphs)[-1])
    print(f"\n=== M8 weight sweep: {pair[0]} -> {pair[1]} ===")
    print(f"  {'w':>5s} {'peri_dev':>10s} {'round_dev':>10s} {'avg_gl':>9s} "
          f"{'IoU':>6s} {'topo':>5s} {'regions':>8s}")
    for row in ef.sweep_weights(graphs, seeds, expected, pair=pair, steps=11,
                                params=VERIFY_PARAMS):
        if "error" in row:
            print(f"  {row['w']:5.2f}  ERROR: {row['error']}")
            continue
        print(f"  {row['w']:5.2f} {row.get('S7_peri_dev_mean_signed', 0):+10.4f} "
              f"{row.get('S7_round_dev_max', 0):10.4f} "
              f"{row.get('S7_avg_gl_dev_mean', 0):+9.4f} "
              f"{str(row.get('S6_iou_area_weighted')):>6s} "
              f"{str(row.get('S7_topo_delta')):>5s} {row.get('n_regions', 0):8d}")
    print("  peri_dev should bottom out at w=0.5 and return toward 0 at the ends.")
    print("  topo must be 0 at every w -- it is a design guarantee, not a metric.")

# --- 3) region registry ---------------------------------------------------------------
# Three sources of region id, all tracked: template (DB09), expert (a closed path with no
# code -> 9001+), unlabeled (a face the code-set rule could not decide -> 8001+, a QA gap).
reg = ef.load_registry(REGISTRY_PATH)
P = FUSION_PARAMS
print("\n=== REGION REGISTRY ===")
if not reg["slices"]:
    print(f"registry empty ({REGISTRY_PATH})"
          + (" -- expected with build_polygons=False." if not P.build_polygons
             else " -- run CELL 9 first."))
else:
    rows = sorted(int(k) for k in reg["slices"])
    tot_unl, tot_new = [], []
    print(f"{'slice':>6s} {'active':>7s} {'template':>9s} {'expert':>16s} "
          f"{'UNLABELED':>16s} {'inactive':>10s}")
    base_gone = {}
    for s in rows:
        rr  = reg["slices"][str(s)]["regions"]
        act = {int(r): v for r, v in rr.items() if v.get("status") == "active"}
        tpl = sorted(r for r in act if r < P.unlabeled_id_start)
        unl = sorted(r for r in act if P.unlabeled_id_start <= r < P.new_id_start)
        new = sorted(r for r in act if r >= P.new_id_start)
        ina = ef.registry_inactive(reg, s)
        tot_unl += unl; tot_new += new
        # BASE ROLLUP: a base region counts as gone on this slice only when EVERY fragment
        # of it is inactive. DELETED is a base-level verdict; individual fragment records
        # going inactive is normal partial coverage.
        bs = ef.registry_base_status(reg, s)
        gone = sorted(b for b, st in bs.items() if st != "active")
        for b in gone:
            base_gone.setdefault(b, []).append(s)
        if unl or new or len(rows) <= 40:
            print(f"{s:6d} {len(act):7d} {len(tpl):9d} {str(new):>16s} "
                  f"{str(unl):>16s} {len(ina):10d}"
                  + (f"  base-gone {gone[:6]}" if gone else ""))
    print(f"\n  {len(rows)} slices tracked   next_new_id = {reg.get('next_new_id')}   "
          f"next_unlabeled_id = {reg.get('next_unlabeled_id')}")
    print(f"  registry file: {REGISTRY_PATH}")
    _everywhere = sorted(b for b, ss in base_gone.items() if len(ss) == len(rows))
    if _everywhere:
        print(f"  !! base region(s) {_everywhere} have NO active fragment on ANY tracked "
              f"slice -- the ONLY candidates for a 'deleted' verdict.")   # TROUBLESHOOTING PRINT
    if tot_unl:
        print(f"\n  !! {len(set(tot_unl))} UNLABELED region id(s) across the run. Each means "
              f"polygonize produced a face whose bounding arcs share no common code, i.e. a "
              f"separating boundary is missing. Fix it in the QA:")
        print("     1. CELL 11's render shows it MAGENTA.")
        print("     2. CELL 9 printed its seeds_inside -- those are the regions that merged.")
        print("     3. Redraw the missing boundary, re-export, re-run CELL 9.")

# Housekeeping is MANUAL ONLY -- a region absent from one slice is normal, not an error:
#   reg, n = ef.registry_purge(reg, slice_index=None)
#   ef.save_registry(reg, REGISTRY_PATH)


In [ ]:
# =====================================================================================
# 6.2 / 6.3  RASTERISE THE ATLAS AND WRITE M3C's INPUT FORMAT
#
# 6.2.1  template grid and affine; 6.2.2 empty int16 volume, background 0
# 6.2.3  per slice: flatten multipolygons, sort faces by descending area, fill
#        exteriors, zero interiors, let nested regions overwrite
# 6.2.5  orientation verified on a representative slice (IoU vs the template)
# 6.3.1  the in-plane grid must be square; 6.3.2 header dim / interval / FOV
# 6.3.4  copied to local disk before M3C reads it
# =====================================================================================
import os
import numpy as np, nibabel as nib
from shapely.geometry import Polygon, MultiPolygon
from skimage.draw import polygon as _sk_polygon

# drift apart. xy_spacing is M3C's marching-cube STEP (2 = 4x fewer triangles); cap=0 drops
# the flat cut-plane face where the volume ends.

# 6.2.3.1  flatten every multi-polygon into a single list of polygons
def _iter_polys(geom):
    if geom is None or geom.is_empty:
        return
    if isinstance(geom, MultiPolygon):
        for g in geom.geoms:
            if not g.is_empty: yield g
    elif isinstance(geom, Polygon):
        yield geom

# 6.2.3  per-slice rasterisation
def rasterize_regions(regions, shape_rc, dtype=np.int16):
    """{rid: [Polygon|MultiPolygon]} -> label image img[row=L-R, col=S-I] = rid."""
    H, W = shape_rc
    img = np.zeros((H, W), dtype=dtype)
    faces = [(p.area, int(rid), p)
             for rid, gs in regions.items()
             for g in (gs if isinstance(gs, (list, tuple)) else [gs])
             for p in _iter_polys(g)]
    # 6.2.3.2  sort faces by descending area
    faces.sort(key=lambda t: t[0], reverse=True)
    for _, rid, poly in faces:
        ext = np.asarray(poly.exterior.coords, float)
        # 6.2.3.3 / 6.2.3.5  fill the exterior ring with the region id; nested faces painted later overwrite it
        rr, cc = _sk_polygon(ext[:, 1], ext[:, 0], shape=(H, W)); img[rr, cc] = rid
        # 6.2.3.4  set interior rings to zero
        for ring in poly.interiors:
            r = np.asarray(ring.coords, float)
            hr, hc = _sk_polygon(r[:, 1], r[:, 0], shape=(H, W)); img[hr, hc] = 0
    return img

def _pick_region_source():
    """Prefer the mirrored full-brain faces; fall back to the single-hemisphere fusion."""
    full = load_state("atlas_regions_full", default={}) or {}
    if any(full.values()):
        return ({s: ef.merge_fragment_regions(r) for s, r in full.items()},
                "atlas_regions_full (mirrored, both hemispheres)")
    half = load_state("fused_regions", default={}) or {}
    if any(half.values()):
        print("  !! using fused_regions (SINGLE hemisphere). Run CELL 10 with "
              "build_polygons=True for the mirrored full brain.")
        return ({s: ef.merge_fragment_regions(r) for s, r in half.items()},
                "fused_regions (single hemisphere)")
    return {}, "none"

# 6.2.5  verify orientation on a representative slice (IoU vs the template)
def verify_slice(sidx=None):
    """GATE: rasterize one slice's faces and overlay on the template slice it came from."""
    import matplotlib.pyplot as plt
    src, _ = _pick_region_source()
    if sidx is None:
        sidx = load_state("trace_meta")["slice_index"]
    if sidx not in src:
        sidx = sorted(src)[len(src) // 2]
    lab_vol = nib.load(TEMPLATE_LABELS); D0, D1, D2 = lab_vol.shape
    got   = rasterize_regions(src[sidx], (D0, D2))
    truth = np.take(lab_vol.get_fdata().astype(np.int32), sidx, axis=SLICE_AXIS)
    inter = ((got > 0) & (truth > 0)).sum(); union = ((got > 0) | (truth > 0)).sum()
    iou = inter / max(union, 1)
    print(f"slice {sidx}: brain-mask IoU (atlas vs template) = {iou*100:.1f}%")
    if iou < 0.7:
        print("  !! LOW IoU -- suspect an orientation bug (flip / rotation / L-R swap). "
              "Stop and check before building the volume.")
    fig, ax = plt.subplots(1, 3, figsize=(12, 4))
    ax[0].imshow(truth, cmap="tab20"); ax[0].set_title(f"template labels (slice {sidx})")
    ax[1].imshow(got,   cmap="tab20"); ax[1].set_title("rasterized atlas faces")
    ax[2].imshow(truth > 0, cmap="Greys"); ax[2].imshow(got > 0, cmap="Reds", alpha=0.4)
    ax[2].set_title("overlay (red = atlas)")
    for a in ax: a.axis("off")
    plt.tight_layout(); plt.show()
    return iou

# 6.2.1 / 6.2.2  template grid and affine, then an empty int16 volume with background 0
def build_label_volume():
    """Burn every slice's faces into one int16 volume on the working grid."""
    src, which = _pick_region_source()
    if not any(src.values()):
        raise RuntimeError("No polygon atlas found. Set FUSION_PARAMS.build_polygons=True "
                           "in CELL 0, then re-run CELL 9 (fuse) and CELL 10 (mirror).")
    lab_vol = nib.load(TEMPLATE_LABELS)
    D0, D1, D2 = lab_vol.shape                     # (L-R, coronal, S-I) for SLICE_AXIS=1
    V = np.zeros((D0, D1, D2), np.int16)
    filled = []
    for sidx, regions in sorted(src.items()):
        if not regions:                            # skip slices with no faces
            continue
        # 6.2.4 / 6.2.4.2  write the slice into the volume (row/col -> array axes 0 and 2)
        V[:, sidx, :] = rasterize_regions(regions, (D0, D2))
        filled.append(sidx)
    ids = sorted(int(r) for r in np.unique(V) if r > 0)
    # 6.2.6  unresolved faces carry unlabeled ids into the volume as a stray material
    stray = [r for r in ids if r >= FUSION_PARAMS.unlabeled_id_start]
    print(f"  source        : {which}")
    print(f"  slices filled : {len(filled)} of {D1}")
    print(f"  volume        : {V.shape}   labels present: {len(ids)}")
    if stray:
        print(f"  !! {len(stray)} unresolved id(s) carried into the volume as a stray "
              f"material: {stray[:12]}")
    return V, lab_vol

# 6.3  write M3C's SPR/SDT input
def write_spr_sdt(V, lab_vol, out_dir=VOL_DIR, name=VOL_NAME, slice_axis=SLICE_AXIS):
    """Write the SPR/SDT pair in the byte order M3C's ReadSdtFile expects."""
    assert slice_axis == 1, "this writer assumes coronal slices on axis 1"
    D0, D1, D2 = V.shape
    # 6.3.1  assert the in-plane grid is square
    if D0 != D2:
        raise ValueError(f"M3C requires a SQUARE in-plane grid (nRow == nCol); got {D0}x{D2}. "
                         f"The '_sq' template should already be square -- check TEMPLATE_LABELS.")
    z0, z1, z2 = [float(z) for z in lab_vol.header.get_zooms()[:3]]
    # 6.3.2  header: 6.3.2.1 dim, 6.3.2.2 interval, 6.3.2.3 FOV
    spr = (f"numDim: 4\n"
           f"dim: {D0} {D2} {D1} 1\n"
           f"fov: {z2*D2:.6f} {z0*D0:.6f} {z1*D1:.6f} 1.0\n"
           f"interval: {z2:.6f} {z0:.6f} {z1:.6f} 1.0\n"
           f"dataType: WORD\n"
           f"sdtOrient: axsi\n"
           f"endian: ieee-le\n")
    # SDT disk order: slice(k) outer, row (axis 0) middle, col (axis 2) inner.
    disk = np.ascontiguousarray(np.transpose(V, (1, 0, 2)).astype('<i2'))
    spr_path, sdt_path = f"{out_dir}/{name}.spr", f"{out_dir}/{name}.sdt"
    # 6.3.3  binary .sdt
    open(spr_path, "w").write(spr); disk.tofile(sdt_path)
    print("\n".join("  " + ln for ln in spr.strip().splitlines()))
    print(f"\n  wrote {spr_path}  ({os.path.getsize(spr_path)} B)")
    print(f"  wrote {sdt_path}  ({os.path.getsize(sdt_path)} B == {D0}*{D2}*{D1}*2)")
    return spr_path, sdt_path

verify_slice()                                     # orientation gate
V, lab_vol = build_label_volume()
spr_path, sdt_path = write_spr_sdt(V, lab_vol)
report_outputs("CELL 13 label volume + SPR/SDT", files=[spr_path, sdt_path])

# CELL 14 pipes these answers automatically; the list is here for a manual M3C run.
_rng = "0" if M3C_SLICE_RANGE is None else f"1 then {M3C_SLICE_RANGE[0]} {M3C_SLICE_RANGE[1]}"
print("\n=== M3C interactive prompts, in order ===")
print(f"  sdt/spr file path                : {VOL_DIR}")
print(f"  sdt/spr file name (no extension) : {VOL_NAME}")
print(f"  specify slice range? (0/1)       : {_rng}   (0 = all {V.shape[1]} slices)")
print(f"  capped? (0/1)                    : {M3C_CAP}")
print(f"  linearize? (0/1)                 : {M3C_LINEARIZE}")
print(f"  centered? (0/1)                  : {M3C_CENTER}")
print(f"  X-Y spacing                      : {M3C_XY_SPACING}")
print(f"  Z spacing                        : {M3C_Z_SPACING}")
print(f"  output Vtk file name             : {VOL_NAME}")


In [ ]:
# =====================================================================================
# 6.4  MULTIPLE-MATERIAL MARCHING CUBES
#
# 6.4.1  build for the host platform; 6.4.2 read SPR/SDT
# 6.4.3  M3C parameters (slice range, cap, linearize, centre, spacing) from 0.4.7
# 6.4.4  march a 2x2x2 cube, classify corners by region id, emit from the case table
# 6.4.5  one PolyData surface (.vtk) per material, plus combined node/element files
# =====================================================================================
import os, glob, shutil, subprocess, zipfile, time

# 6.4.1  locate the M3C source tree
def _find_m3c_source():
    """Accept a zip or an already-extracted folder. Returns the dir holding main.cxx."""
    cands = [M3C_ZIP, f"{DRIVE_ROOT}/M3C2012", f"{DRIVE_ROOT}/m3c"]
    cands += sorted(glob.glob(f"{DRIVE_ROOT}/**/M3C2012*", recursive=True))[:5]
    for c in cands:
        if c.endswith(".zip") and os.path.exists(c):
            ex = "/content/m3c_src"
            if not os.path.exists(ex):
                with zipfile.ZipFile(c) as z: z.extractall(ex)
            hits = glob.glob(f"{ex}/**/main.cxx", recursive=True)
            if hits: return os.path.dirname(hits[0])
        elif os.path.isdir(c):
            hits = glob.glob(f"{c}/**/main.cxx", recursive=True)
            if hits: return os.path.dirname(hits[0])
    raise FileNotFoundError(f"M3C source not found. Put the zip at {M3C_ZIP} or extract it "
                            f"under {DRIVE_ROOT}/M3C2012.")

# 6.4.1  build M3C for the host platform
def build_m3c(force=False):
    """Patch + compile once. Returns the path to the binary."""
    exe = f"{M3C_BUILD}/m3c"
    if os.path.exists(exe) and not force:
        print(f"m3c already built -> {exe}")
        return exe
    src = _find_m3c_source()
    print(f"M3C source: {src}")
    if os.path.exists(M3C_BUILD): shutil.rmtree(M3C_BUILD)
    shutil.copytree(src, M3C_BUILD)
    cwd = os.getcwd(); os.chdir(M3C_BUILD)
    try:
        for f in glob.glob("*.cxx") + glob.glob("*.h"):              # CRLF -> LF
            b = open(f, "rb").read().replace(b"\r\n", b"\n")
            open(f, "wb").write(b)
        for lower, cased in (("cnimmc.h", "cniMMC.h"),
                             ("fileoperation.h", "FileOperation.h")):
            if os.path.exists(lower) and not os.path.exists(cased):
                shutil.copy(lower, cased)                            # case-sensitive FS
        for f in glob.glob("*.cxx"):
            s = open(f).read()
            if '#include "iostream.h"' in s:
                open(f, "w").write(s.replace('#include "iostream.h"',
                                             "#include <iostream>\nusing namespace std;"))
        s = open("vmatrix.h").read()
        if not s.lstrip().startswith('#include "cnHeader.h"'):
            open("vmatrix.h", "w").write('#include "cnHeader.h"\n' + s)
        open("conio.h", "w").write(
            "#ifndef _CONIO_H_SHIM\n#define _CONIO_H_SHIM\n#include <cstdio>\n"
            "static inline int getch(void){ return getchar(); }\n#endif\n")
        open("m3c_compat.h", "w").write(
            "#ifndef M3C_COMPAT_H\n#define M3C_COMPAT_H\n"
            "#include <cstring>\n#include <cstdio>\n#include <cstdlib>\n"
            "#ifndef _WIN32\n  #define _ecvt ecvt\n#endif\n#endif\n")
        s = open("main.cxx").read()
        s = s.replace("\tgets(sdtPath);",
                      '\tif (!fgets(sdtPath, sizeof(sdtPath), stdin)) return -1;\n'
                      '\tsdtPath[strcspn(sdtPath, "\\r\\n")] = 0;')
        s = s.replace("\tgets(sdtName);",
                      '\tif (!fgets(sdtName, sizeof(sdtName), stdin)) return -1;\n'
                      '\tsdtName[strcspn(sdtName, "\\r\\n")] = 0;')
        open("main.cxx", "w").write(s)
        s = open("cniMMC.cxx").read()
        n = (s.count('strcat(fn_sdt, "\\\\");') + s.count('strcat(fnout_nod,"\\\\");')
             + s.count('strcat(fnout_elm,"\\\\");'))
        s = (s.replace('strcat(fn_sdt, "\\\\");',   'strcat(fn_sdt, "/");')
              .replace('strcat(fnout_nod,"\\\\");', 'strcat(fnout_nod,"/");')
              .replace('strcat(fnout_elm,"\\\\");', 'strcat(fnout_elm,"/");'))
        open("cniMMC.cxx", "w").write(s)
        print(f"  patched {n} Windows path separator(s)")
        srcs = [f for f in sorted(glob.glob("*.cxx")) if "CniMMCCmds" not in f]
        cmd  = ["g++", "-O2", "-w", "-DCNISTATIC", "-include", "m3c_compat.h", "-I.",
                "-o", "m3c"] + srcs
        print("  compiling:", " ".join(os.path.basename(f) for f in srcs))
        r = subprocess.run(cmd, capture_output=True, text=True)
        if r.returncode != 0 or not os.path.exists("m3c"):
            print(r.stdout[-3000:]); print(r.stderr[-3000:])
            raise RuntimeError("M3C failed to compile -- see the output above")
    finally:
        os.chdir(cwd)
    print(f"BUILD OK -> {exe}")
    return exe

# 6.4.2 - 6.4.5  read SPR/SDT, set parameters, march the 2x2x2 cube, write one surface per material
def run_m3c(spr=None, sdt=None, name=None, cap=M3C_CAP, linearize=M3C_LINEARIZE,
            center=M3C_CENTER, xy_spacing=M3C_XY_SPACING, z_spacing=M3C_Z_SPACING,
            slice_range=M3C_SLICE_RANGE, timeout=7200):
    """Feed CELL 13's SPR/SDT to M3C and collect the per-region .vtk files.
    slice_range=(start, end) uses M3C's 1-based numbering; None = all slices."""
    exe  = build_m3c()
    name = name or VOL_NAME
    spr  = spr or f"{VOL_DIR}/{name}.spr"
    sdt  = sdt or f"{VOL_DIR}/{name}.sdt"
    for p in (spr, sdt):
        if not os.path.exists(p):
            raise FileNotFoundError(f"{p} missing -- run CELL 13 first")
    # Stage locally: M3C reads the whole .sdt, and reading it across Drive is slow.
    os.makedirs(M3C_RUN, exist_ok=True)
    for p in (spr, sdt):
        dst = f"{M3C_RUN}/{os.path.basename(p)}"
        if not (os.path.exists(dst) and os.path.getsize(dst) == os.path.getsize(p)):
            shutil.copy(p, dst)
    for old in glob.glob(f"{M3C_RUN}/*.vtk"):
        os.remove(old)

    ans = [M3C_RUN, name]
    ans += (["1", str(slice_range[0]), str(slice_range[1])] if slice_range else ["0"])
    ans += [str(cap), str(linearize), str(center), str(xy_spacing), str(z_spacing), name]
    print("piping answers:", " | ".join(ans))

    t0 = time.time()
    r = subprocess.run([exe], input="\n".join(ans) + "\n", cwd=M3C_RUN,
                       capture_output=True, text=True, timeout=timeout)
    tail = (r.stdout or "")[-1500:]
    print(tail)
    if "failure" in tail.lower() or "Error" in tail:
        print("!! M3C reported a problem -- see the message above")

    vtks = sorted(glob.glob(f"{M3C_RUN}/{name}_*.vtk"))
    print(f"\n{len(vtks)} surface(s) in {time.time()-t0:.1f}s")
    for v in vtks[:10]:
        print(f"   {os.path.basename(v):40s} {os.path.getsize(v)/1e6:7.2f} MB")
    if len(vtks) > 10:
        print(f"   ... and {len(vtks)-10} more")
    if not vtks:
        raise RuntimeError("M3C produced no .vtk files. Most common causes: the SPR/SDT pair "
                           "is not in M3C_RUN, or the volume has no non-zero labels.")
    if COPY_VTK_TO_DRIVE:
        os.makedirs(VTK_OUT, exist_ok=True)
        for v in vtks:
            shutil.copy(v, f"{VTK_OUT}/{os.path.basename(v)}")
        print(f"\ncopied to {VTK_OUT}")
    return vtks

# Settings come from CELL 13 (M3C_CAP, M3C_XY_SPACING, ...) so the printed prompt guide and
# this automated run cannot disagree. Reduce mesh density via M3C_XY_SPACING, never by
# decimating afterwards.
vtk_files = run_m3c()


In [ ]:
# =====================================================================================
# 6.5 / 6.6  SURFACE CONDITIONING AND OUTPUT DELIVERABLES
#
# 6.5.1  read the VTK files; 6.5.2 Taubin smoothing (volume preserving)
# 6.6.1  NIfTI label-map on the template affine; 6.6.2 region look-up table
# 6.6.5  per-region STL and OBJ, merged interactive mesh, standalone Plotly viewer
# =====================================================================================
import os, re, glob, csv, struct, time, shutil
import numpy as np, nibabel as nib

for d in (NIFTI_DIR, MESH_DIR, VIEWER_DIR):
    os.makedirs(d, exist_ok=True)

_PAL = {}
if os.path.exists(PALETTE_CSV):
    for r in csv.DictReader(open(PALETTE_CSV, newline="")):
        _PAL[r["name"].strip().lower()] = [int(r["r"]), int(r["g"]), int(r["b"])]
    print(f"palette: {len(_PAL)} reference colours loaded")

# 6.6.2  region look-up table: name
def _region_name(rid):
    e = combined_lut.get(int(rid), {})
    return e.get("abbrev", str(rid)), e.get("name", e.get("abbrev", str(rid)))

# 6.6.2  region look-up table: colour
def _rgb255(rid):
    _, nm = _region_name(rid)
    hit = _PAL.get(nm.strip().lower())
    if hit: return hit
    c = region_color(int(rid))
    if isinstance(c, str):
        c = c.lstrip("#"); return [int(c[0:2],16), int(c[2:4],16), int(c[4:6],16)]
    return [int(round(v)) for v in c[:3]]

# 6.6.1  NIfTI label-map on the template affine
def export_nifti(volume, name="macaque_atlas_labels"):
    """Label-map on the working-grid affine, plus its region look-up table."""
    tpl = nib.load(TEMPLATE_LABELS)
    img = nib.Nifti1Image(volume.astype(np.int16), tpl.affine, tpl.header)
    img.set_data_dtype(np.int16)
    out = f"{NIFTI_DIR}/{name}.nii.gz"; nib.save(img, out)
    ids = sorted(int(r) for r in np.unique(volume) if r > 0)
    with open(f"{NIFTI_DIR}/{name}_LUT.csv", "w") as f:
        f.write("id,abbreviation,name,color_hex,n_voxels\n")
        for rid in ids:
            ab, nm = _region_name(rid); r, g, b = _rgb255(rid)
            f.write(f'{rid},"{ab}","{nm}",#{r:02x}{g:02x}{b:02x},{int((volume==rid).sum())}\n')
    print(f"NIfTI  : {out}  ({os.path.getsize(out)/1e6:.1f} MB, {len(ids)} regions)")
    return out

# 6.5.1  read the VTK files
def read_vtk_polydata(path):
    """Minimal ASCII legacy VTK PolyData reader (points + triangles)."""
    txt = open(path).read()
    mp = re.search(r"POINTS\s+(\d+)\s+\w+\s*\n", txt)
    if not mp:
        return np.zeros((0, 3)), np.zeros((0, 3), int)
    n = int(mp.group(1))
    Vt = np.fromstring(" ".join(txt[mp.end():].split("\n")[:n]), sep=" ").reshape(n, 3)
    mf = re.search(r"POLYGONS\s+(\d+)\s+(\d+)\s*\n", txt)
    if not mf:
        return Vt, np.zeros((0, 3), int)
    body = txt[mf.end():].split("\n")[:int(mf.group(1))]
    F = np.array([[int(x) for x in ln.split()[1:4]] for ln in body if ln.strip()], int)
    return Vt, F

# 6.5.2  Taubin smoothing (volume preserving)
def taubin_smooth(Vt, F, iterations=SMOOTH_ITERS, lam=0.5, mu=-0.53):
    """Volume-preserving smoothing: a shrinking lambda pass alternated with an inflating
    mu pass. Tiny nuclei get fewer passes so they are not smoothed away."""
    if iterations <= 0 or len(F) == 0:
        return Vt
    it_n = iterations if len(F) >= 500 else max(5, iterations // 4)
    e = np.vstack([F[:, [0,1]], F[:, [1,2]], F[:, [2,0]]]); e = np.vstack([e, e[:, ::-1]])
    src, dst = e[:, 0], e[:, 1]; n = len(Vt)
    cnt = np.bincount(src, minlength=n).astype(float); cnt[cnt == 0] = 1
    Vs = Vt.astype(np.float64).copy()
    for it in range(it_n * 2):
        w = lam if it % 2 == 0 else mu
        nb = np.stack([np.bincount(src, weights=Vs[dst, d], minlength=n) for d in range(3)], 1)
        Vs = Vs + w * (nb / cnt[:, None] - Vs)
    return Vs

# 6.6.5.1  per-region OBJ
def _write_obj(p, Vt, F):
    with open(p, "w") as f:
        f.write("\n".join(f"v {x:.5f} {y:.5f} {z:.5f}" for x, y, z in Vt) + "\n")
        f.write("\n".join(f"f {a+1} {b+1} {c+1}" for a, b, c in F) + "\n")

# 6.6.5.1  per-region STL
def _write_stl(p, Vt, F):
    tri = Vt[F]; nrm = np.cross(tri[:, 1]-tri[:, 0], tri[:, 2]-tri[:, 0])
    ln = np.linalg.norm(nrm, axis=1, keepdims=True); nrm = nrm / np.where(ln == 0, 1, ln)
    with open(p, "wb") as f:
        f.write(b"\0" * 80); f.write(struct.pack("<I", len(F)))
        for m in range(len(F)):
            f.write(struct.pack("<12fH", *nrm[m], *tri[m,0], *tri[m,1], *tri[m,2], 0))

# 6.5  read and condition every region surface
def load_meshes(vtk_dir=None, smooth=SMOOTH_ITERS):
    """Read every per-region .vtk, Taubin-smooth it, and write STL + OBJ per region."""
    vtk_dir = vtk_dir or (VTK_OUT if os.path.isdir(VTK_OUT) else M3C_RUN)
    files = sorted(glob.glob(f"{vtk_dir}/*_*.vtk"))
    if not files:
        raise FileNotFoundError(f"no .vtk in {vtk_dir} -- run CELL 14 first")
    meshes, tot = {}, 0
    for p in files:
        m = re.search(r"_(\d+)\.vtk$", os.path.basename(p))
        if not m: continue
        rid = int(m.group(1)); Vt, F = read_vtk_polydata(p)
        if len(F) == 0: continue
        Vt = taubin_smooth(Vt, F, smooth)
        meshes[rid] = (Vt, F); tot += len(F)
        ab, _ = _region_name(rid)
        stem = f"{MESH_DIR}/{re.sub(r'[^A-Za-z0-9_-]', '_', ab)}_{rid}"
        _write_stl(f"{stem}.stl", Vt, F); _write_obj(f"{stem}.obj", Vt, F)
    print(f"Meshes : {len(meshes)} regions, {tot:,} triangles (Taubin x{smooth}) -> {MESH_DIR}")
    return meshes

# 6.6.5.3 / 6.6.5.5  merge all regions into one mesh and write the standalone Plotly viewer
def build_viewer(meshes, out_name="macaque_atlas_3D"):
    """One merged Mesh3d with per-vertex colour and per-vertex region name."""
    import plotly.graph_objects as go
    X, Y, Z, I, J, K, VC, HT = [], [], [], [], [], [], [], []
    off = 0
    for rid in sorted(meshes):
        Vt, F = meshes[rid]; _, nm = _region_name(rid)
        X.append(Vt[:, 0].astype(np.float32)); Y.append(Vt[:, 1].astype(np.float32))
        Z.append(Vt[:, 2].astype(np.float32))
        I.append((F[:, 0] + off).astype(np.int32)); J.append((F[:, 1] + off).astype(np.int32))
        K.append((F[:, 2] + off).astype(np.int32))
        VC.append(np.tile(np.array(_rgb255(rid), np.uint8), (len(Vt), 1)))
        HT.extend([nm] * len(Vt)); off += len(Vt)
    X = np.concatenate(X); Y = np.concatenate(Y); Z = np.concatenate(Z)
    I = np.concatenate(I); J = np.concatenate(J); K = np.concatenate(K)
    VC = np.concatenate(VC)
    print(f"  merged: {len(X):,} verts, {len(I):,} tris, {len(meshes)} regions")
    fig = go.Figure(go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K,
        vertexcolor=VC, hovertext=HT, hoverinfo="text", flatshading=False,
        lighting=dict(ambient=0.5, diffuse=0.85, fresnel=0.1, roughness=0.4, specular=0.3),
        lightposition=dict(x=100, y=200, z=300)))
    fig.update_layout(
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False),
                   zaxis=dict(visible=False), bgcolor="black", aspectmode="data"),
        margin=dict(l=0, r=0, t=30, b=0), paper_bgcolor="black",
        title=dict(text=VIEWER_TITLE, x=0.5, font=dict(color="white", size=14)))
    tmp = f"/content/{out_name}.html"
    fig.write_html(tmp, include_plotlyjs="cdn", full_html=True, config={"responsive": True})
    out = f"{VIEWER_DIR}/{out_name}.html"; shutil.copy(tmp, out)
    mb = os.path.getsize(out) / 1e6
    print(f"Viewer : {out}  ({mb:.1f} MB)")
    if mb > SIZE_LIMIT_MB:
        print(f"  !! over {SIZE_LIMIT_MB} MB -- re-run CELL 14 with run_m3c(xy_spacing=3, cap=0)")
    if SHOW_INLINE:
        fig.show()
    return fig, out

t0 = time.time()
print(f"=== DELIVERABLES -> {FINAL_DIR} ===\n")
nii    = export_nifti(V)
meshes = load_meshes()
fig, viewer_html = build_viewer(meshes)
report_outputs("CELL 15 deliverables", files=[nii, viewer_html])
print(f"\nDone in {time.time()-t0:.1f}s")


In [ ]:
# =====================================================================================
# 6.6.3 / 6.6.4  LABEL DESCRIPTION FILES
#
# 6.6.3  ITK-SNAP label description file
# 6.6.4  FreeSurfer look-up table (FreeView, 3D Slicer)
# =====================================================================================
import os, csv

# 6.6.3 / 6.6.4  ITK-SNAP label description file and FreeSurfer look-up table
def export_label_files(nifti_dir=None, stem="macaque_atlas_labels"):
    nifti_dir = nifti_dir or NIFTI_DIR
    lut_csv = f"{nifti_dir}/{stem}_LUT.csv"
    if not os.path.exists(lut_csv):
        raise FileNotFoundError(f"{lut_csv} not found -- run CELL 15 (export_nifti) first")

    rows = []
    for r in csv.DictReader(open(lut_csv, newline="")):
        hx = r["color_hex"].lstrip("#")
        rows.append((int(r["id"]),
                     int(hx[0:2], 16), int(hx[2:4], 16), int(hx[4:6], 16),
                     r["name"].replace('"', "'").strip(),
                     r["abbreviation"].replace('"', "'").strip()))
    rows.sort(key=lambda t: t[0])

    itk = f"{nifti_dir}/{stem}_ITKSNAP.label"
    with open(itk, "w") as f:
        f.write("################################################\n"
                "# ITK-SnAP Label Description File\n"
                "# IDX   -R-  -G-  -B-  -A--  VIS MSH  LABEL\n"
                "################################################\n")
        f.write('    0    0    0    0  0.00   0  0    "Clear Label"\n')
        for rid, r, g, b, nm, _ab in rows:
            f.write(f'{rid:5d}  {r:3d}  {g:3d}  {b:3d}  1.00   1  1    "{nm}"\n')

    fs = f"{nifti_dir}/{stem}_FreeSurferLUT.txt"
    with open(fs, "w") as f:
        f.write("# Atlas colour LUT -- FreeSurfer format (FreeView, 3D Slicer)\n")
        f.write("# No.  Label Name       R   G   B   A\n")
        f.write(f"{0:<6d}{'Unknown':<40s}{0:4d}{0:4d}{0:4d}{0:4d}\n")
        for rid, r, g, b, nm, ab in rows:
            safe = (ab or nm).replace(" ", "-")[:38]
            f.write(f"{rid:<6d}{safe:<40s}{r:4d}{g:4d}{b:4d}{0:4d}\n")

    print(f"Labels : {len(rows)} regions")
    print(f"  ITK-SNAP   : {itk}")
    print(f"  FreeSurfer : {fs}")
    print("\nTo view in ITK-SNAP (download these three first):")
    _sid = next(iter(load_state("regs", default={})), None)
    if _sid:
        print(f"  1. File > Open Main Image...           {SUBJ_DIR}/{_sid}_in_subjgrid.nii.gz")
    print(f"  2. Segmentation > Open Segmentation...  {nifti_dir}/{stem}.nii.gz")
    print(f"  3. Segmentation > Import Label Desc...  {os.path.basename(itk)}")
    return itk, fs

export_label_files()


In [ ]:
# =====================================================================================
# 6.6.6  3D SLICER EXPORT + INLINE PREVIEW
#
# 6.6.6.1  per-region VTK, reusing the meshes conditioned at 6.5
# 6.6.6.2  models.csv listing region id, name, colour and file
# 6.6.6.3  Slicer loader script that reads models.csv and builds the scene
# =====================================================================================
import os, glob, time, csv
import numpy as np

# 6.6.6.1  per-region VTK PolyData
def write_vtk_polydata(path, Vt, F, title="atlas region"):
    P = np.ascontiguousarray(Vt, dtype='>f4')
    C = np.empty((len(F), 4), dtype='>i4'); C[:, 0] = 3; C[:, 1:] = F
    with open(path, "wb") as f:
        f.write(b"# vtk DataFile Version 3.0\n")
        f.write((title[:250] + "\n").encode())
        f.write(b"BINARY\nDATASET POLYDATA\n")
        f.write(f"POINTS {len(P)} float\n".encode()); f.write(P.tobytes()); f.write(b"\n")
        f.write(f"POLYGONS {len(F)} {C.size}\n".encode()); f.write(C.tobytes()); f.write(b"\n")

# 6.6.6  cluster decimation for the inline preview
def cluster_decimate(Vt, F, cell):
    """Vertex clustering that preserves winding order."""
    if cell <= 0 or len(F) == 0:
        return Vt, F
    key = np.floor(Vt / cell).astype(np.int64)
    uniq, inv = np.unique(key, axis=0, return_inverse=True)
    cnt = np.bincount(inv, minlength=len(uniq)).astype(float)
    nV = np.stack([np.bincount(inv, weights=Vt[:, d], minlength=len(uniq)) / cnt
                   for d in range(3)], axis=1)
    nf = inv[F]
    nf = nf[(nf[:,0] != nf[:,1]) & (nf[:,1] != nf[:,2]) & (nf[:,0] != nf[:,2])]
    if len(nf) == 0:
        return nV, nf
    _, idx = np.unique(np.sort(nf, axis=1), axis=0, return_index=True)
    return nV, nf[np.sort(idx)]

# 6.6.6.2 / 6.6.6.3  models.csv and the Slicer loader script
def export_for_slicer(meshes, out_dir=SLICER_DIR):
    """One .vtk per region + a colour table + a paste-into-Slicer loader script."""
    t0, rows, tot = time.time(), [], 0
    for rid in sorted(meshes):
        Vt, F = meshes[rid]
        if len(F) == 0: continue
        ab, nm = _region_name(rid); r, g, b = _rgb255(rid)
        stem = f"{''.join(c if c.isalnum() or c in '-_' else '_' for c in ab)}_{rid}"
        write_vtk_polydata(f"{out_dir}/{stem}.vtk", Vt, F, title=nm)
        rows.append((f"{stem}.vtk", rid, ab, nm, r, g, b)); tot += len(F)
    with open(f"{out_dir}/models.csv", "w", newline="") as f:
        w = csv.writer(f)
        w.writerow(["file", "id", "abbrev", "name", "r", "g", "b"]); w.writerows(rows)
    loader = '''# Paste this whole block into 3D Slicer:  View > Python Console
import os, csv, slicer
D = os.path.dirname(os.path.abspath("models.csv")) if os.path.exists("models.csv") else \\
    r"PASTE_THE_FOLDER_PATH_HERE"
with open(os.path.join(D, "models.csv"), newline="") as f:
    rows = list(csv.DictReader(f))
for i, row in enumerate(rows):
    node = slicer.util.loadModel(os.path.join(D, row["file"]))
    if node is None:
        print("failed:", row["file"]); continue
    node.SetName(row["name"])
    d = node.GetDisplayNode()
    d.SetColor(int(row["r"])/255.0, int(row["g"])/255.0, int(row["b"])/255.0)
    d.SetOpacity(1.0)
    d.SetInterpolation(1)     # Gouraud, to match the Plotly viewer
    d.SetAmbient(0.25); d.SetDiffuse(0.80); d.SetSpecular(0.20); d.SetPower(15)
    d.SetBackfaceCulling(0)
    print(f"{i+1}/{len(rows)}  {row['name']}")
print("done -- open the 3D view and use the eye icon to toggle regions")
'''
    open(f"{out_dir}/load_in_slicer.py", "w").write(loader)
    mb = sum(os.path.getsize(p) for p in glob.glob(f"{out_dir}/*.vtk")) / 1e6
    print(f"Slicer : {len(rows)} .vtk models, {tot:,} triangles, {mb:.1f} MB -> {out_dir}")
    print(f"         + models.csv (colour table) + load_in_slicer.py")
    print(f"         took {time.time()-t0:.1f}s")
    return out_dir

# 6.6.5.4  inline preview and visual parameters
def preview_inline(meshes, target=PREVIEW_TRIS, cell=PREVIEW_SCALE):
    """Decimated inline render -- a proxy for the full-res .html."""
    import plotly.graph_objects as go
    tot = sum(len(F) for _, F in meshes.values())
    if cell is None:
        cell = 0.0
        if tot > target:
            cell = 0.15
            while cell < 8.0:
                est = sum(len(cluster_decimate(Vt, F, cell)[1]) for Vt, F in meshes.values())
                if est <= target: break
                cell *= 1.4
    X, Y, Z, I, J, K, VC, HT = [], [], [], [], [], [], [], []
    off = 0
    for rid in sorted(meshes):
        Vt, F = meshes[rid]
        if cell > 0:
            Vt, F = cluster_decimate(Vt, F, cell)
        if len(F) == 0: continue
        _, nm = _region_name(rid)
        X.append(Vt[:, 0].astype(np.float32)); Y.append(Vt[:, 1].astype(np.float32))
        Z.append(Vt[:, 2].astype(np.float32))
        I.append((F[:, 0] + off).astype(np.int32)); J.append((F[:, 1] + off).astype(np.int32))
        K.append((F[:, 2] + off).astype(np.int32))
        VC.append(np.tile(np.array(_rgb255(rid), np.uint8), (len(Vt), 1)))
        HT.extend([nm] * len(Vt)); off += len(Vt)
    X = np.concatenate(X); Y = np.concatenate(Y); Z = np.concatenate(Z)
    I = np.concatenate(I); J = np.concatenate(J); K = np.concatenate(K)
    VC = np.concatenate(VC)
    print(f"Preview: {len(I):,} tris (from {tot:,}"
          + (f", cluster {cell:.2f} mm" if cell else "") + ")")
    fig = go.Figure(go.Mesh3d(x=X, y=Y, z=Z, i=I, j=J, k=K,
        vertexcolor=VC, hovertext=HT, hoverinfo="text", flatshading=False,
        lighting=dict(ambient=0.5, diffuse=0.85, fresnel=0.1, roughness=0.4, specular=0.3),
        lightposition=dict(x=100, y=200, z=300)))
    fig.update_layout(
        scene=dict(xaxis=dict(visible=False), yaxis=dict(visible=False),
                   zaxis=dict(visible=False), bgcolor="black", aspectmode="data"),
        margin=dict(l=0, r=0, t=30, b=0), paper_bgcolor="black", height=650,
        title=dict(text="PREVIEW (decimated) - full resolution is in the .html",
                   x=0.5, font=dict(color="white", size=13)))
    fig.show()
    return fig

export_for_slicer(meshes)
preview_inline(meshes)
